# Medical Fact-Checking with RAG: Topic Identification and Truth Prediction

This notebook builds a retrieval-augmented generation (RAG) pipeline for medical fact-checking. Given a medical statement, the system must:

1. **Identify the topic** the statement belongs to (e.g., cardiology, neurology)
2. **Predict whether the statement is true or false** using evidence from a knowledge base

We take a systematic approach: first we set up the infrastructure, then benchmark multiple retrieval and reranking strategies, and finally run ablation studies to find the optimal configuration.

The pipeline has three core components:
- **Embedding model** (ModernBERT) for dense retrieval from a ChromaDB vector store
- **Cross-encoder reranker** (MiniLM) for rescoring candidate passages
- **LLM** (Qwen3-4B) for final reasoning over the top passages

# AI USAGE DISCLOSURE

In order to be as academically transparent as possible, we note that Artificial Intelligence (AI) tools like ChatGPT and Claude were used in this project to support all stages of work; brainstorming, generating and debugging code, and in writing this report. All AI-generated output was reviewed and adapted by the authors.

## 1. Environment Setup

We start by installing dependencies and importing libraries. PyTorch is configured with CUDA support and TF32 precision for faster GPU computation.

In [ ]:
%pip install -q --upgrade pip

# Install PyTorch with CUDA support (use cu124 if your nvidia-smi shows CUDA 12.4+)
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install core dependencies (transformers from source for latest model support e.g. Qwen3.5)
%pip install -q "protobuf<6" "pyarrow" "accelerate>=1.0" "sentence-transformers>=3.0,<4.0" "chromadb>=0.5.5,<0.7" "loguru>=0.7" "matplotlib>=3.7" "pandas>=2.0" "seaborn>=0.13" "scikit-learn>=1.3" "tqdm>=4.66" "hf_xet" "ipywidgets" "jsonlines"
%pip install -q git+https://github.com/huggingface/transformers.git

# Optional fix for telemetry dependency noise
%pip install -q --upgrade opentelemetry-sdk opentelemetry-api opentelemetry-exporter-otlp-proto-http

# For claude
%pip install -q anthropic 
%pip install -q python-dotenv

import os
import gc
import re
import json
import time
import logging
from pathlib import Path

# Disable Chroma telemetry before importing chromadb
os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["CHROMA_TELEMETRY_ENABLED"] = "FALSE"
logging.getLogger("chromadb.telemetry.product.posthog").setLevel(logging.CRITICAL)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from loguru import logger

import chromadb
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

logger.remove()
logger.add(lambda msg: print(msg, end=""), level="INFO")

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("Local setup complete. If you just installed packages, restart the kernel once.")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2026.2.1 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.0,!=4.57.4,!=4.57.5,<=4.57.6,>=4.51.3, but you have transformers 5.3.0.dev0 which is incompatible.
unsloth-zoo 2026.2.1 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.4,!=4.57.5,<=4.57.6,>=4.51.3, but you have transformers 5.3.0.dev0 which is incompatible.
xformers 0.0.35 requires torch>=2.10, but you have torch 2.5.1+cu121 which is incompatible.


In [ ]:
PROJECT_ROOT = Path.cwd()
print(f"Project root: {PROJECT_ROOT}")
if torch.cuda.is_available():
    print(f"CUDA available: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA not detected. The notebook will run on CPU unless GPU drivers/CUDA are configured.")


## 2. Configuration

Here we define the three models that make up our pipeline:

- **Embedding model**: `nomic-ai/modernbert-embed-base` -- a ModernBERT-based encoder that maps text to dense vectors for similarity search
- **Cross-encoder reranker**: `cross-encoder/ms-marco-MiniLM-L-6-v2` -- a lightweight model that scores query-document pairs more accurately than embedding similarity alone
- **LLM**: `Qwen/Qwen3-4B-Instruct-2507` -- a 4B-parameter instruction-tuned model for final reasoning

We also set chunking parameters (512 characters with 64-character overlap) and generation settings. These chunking defaults will be revisited in the ablation study later.

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

LLM_MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
EMBED_MODEL_ID = "nomic-ai/modernbert-embed-base"
CROSS_ENCODER_MODEL_ID = "cross-encoder/ms-marco-MiniLM-L-6-v2"

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path.cwd()

DB_PATH = str(PROJECT_ROOT / "chroma_db")
DATA_PATH = str(PROJECT_ROOT / "data")
COLLECTION_NAME = "medical_topics_modernbert"

CHUNK_SIZE = 512
CHUNK_OVERLAP = 64
MAX_NEW_TOKENS = 96
TEMPERATURE = 0.0

os.makedirs(DB_PATH, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)

logger.info(f"Using device: {DEVICE}\n")
logger.info(
    f"LLM: {LLM_MODEL_ID}\nEmbedding: {EMBED_MODEL_ID}\nReranker: {CROSS_ENCODER_MODEL_ID}\n"
 )


## 2b. Checkpoint System

Every benchmark result is written to disk **immediately** after it is computed, using an append-only JSONL file. If the notebook crashes or is interrupted, re-running any benchmark cell will automatically skip rows that are already saved and resume from where it left off. Nothing is ever lost.

In [ ]:
import jsonlines

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

TOPIC_CKPT         = RESULTS_DIR / "topic_results.jsonl"
TRUTH_CKPT         = RESULTS_DIR / "truth_results.jsonl"
ABLATION_CKPT      = RESULTS_DIR / "ablation_results.jsonl"
LLM_CMP_TOPIC_CKPT = RESULTS_DIR / "llm_cmp_topic_results.jsonl"
LLM_CMP_TRUTH_CKPT = RESULTS_DIR / "llm_cmp_truth_results.jsonl"


class Checkpoint:
    """
    Append-only JSONL checkpoint.

    On init, reads all existing records so done() checks are instant (O(1)).
    write() appends one record immediately to disk -- a crash can only lose
    the single prediction currently in flight.
    to_dataframe() returns everything as a pandas DataFrame.
    delete() wipes the file so you can start a benchmark from scratch.
    """

    def __init__(self, path, key_fields):
        self.path = Path(path)
        self.key_fields = list(key_fields)
        self._done = set()
        self._load()

    def _load(self):
        if not self.path.exists():
            return
        with jsonlines.open(self.path) as reader:
            for obj in reader:
                self._done.add(self._key(obj))
        logger.info(f"[Checkpoint] Loaded {len(self._done)} existing records from {self.path.name}\n")

    def _key(self, obj):
        return tuple(obj[f] for f in self.key_fields)

    def done(self, **kwargs):
        """Return True if this combination of key fields is already saved."""
        return tuple(kwargs[f] for f in self.key_fields) in self._done

    def write(self, record):
        """Append one record to the JSONL file immediately."""
        with jsonlines.open(self.path, mode="a") as writer:
            writer.write(record)
        self._done.add(self._key(record))

    def to_dataframe(self):
        """Load all records into a DataFrame."""
        if not self.path.exists():
            return pd.DataFrame()
        rows = []
        with jsonlines.open(self.path) as reader:
            for obj in reader:
                rows.append(obj)
        return pd.DataFrame(rows)

    def count(self):
        return len(self._done)

    def delete(self):
        """Wipe checkpoint so the benchmark runs from scratch."""
        if self.path.exists():
            self.path.unlink()
        self._done.clear()
        logger.info(f"[Checkpoint] Deleted {self.path.name}\n")


logger.info("Checkpoint class ready. Results directory: " + str(RESULTS_DIR) + "\n")


## 2c. Strategy Rename Map

We adopt a consistent naming convention for all strategies: `{task}_{retrieval}_{pool}_{selection}`. The checkpoint files on disk still use the original names; this mapping is applied at load time so all plots and tables use the new names. The duplicate `rerank_topdoc_k12` strategy (logically identical to `rerank_max_k12`) is filtered out.

In [ ]:
TOPIC_RENAME_MAP = {
    "embed_top1":          "topic_embed_k1_top1",
    "embed_vote_k5":       "topic_embed_k5_vote",
    "embed_vote_k12":      "topic_embed_k12_vote",
    "embed_vote_k25":      "topic_embed_k25_vote",
    "rerank_topdoc_k12":   None,   # duplicate of rerank_max_k12 -- drop
    "rerank_max_k5":       "topic_rerank_k5_top1",
    "rerank_max_k12":      "topic_rerank_k12_top1",
    "rerank_max_k25":      "topic_rerank_k25_top1",
    "rerank_sum_k12":      "topic_rerank_k12_sum",
    "llm_rerank_k12_top3": "topic_llm_rerank_k12_top3",
}

TRUTH_RENAME_MAP = {
    "truth_embed_top3":      "truth_embed_k3",
    "truth_embed_top5":      "truth_embed_k5",
    "truth_rerank_top3":     "truth_rerank_k12_top3",
    "truth_rerank_top5":     "truth_rerank_k12_top5",
    "truth_rerank_top3_k25": "truth_rerank_k25_top3",
    # Claude strategies get the same base name -- model column distinguishes
    "claude_embed_top5":     "truth_embed_k5",
    "claude_embed_top10":    "truth_embed_k10",
}

# Combined map for convenience (no key collisions between the two)
ALL_RENAME_MAP = {**TOPIC_RENAME_MAP, **TRUTH_RENAME_MAP}


def apply_renames(df, rename_map=None, col="strategy"):
    """Apply strategy renames and filter out entries mapped to None."""
    if df.empty or col not in df.columns:
        return df
    if rename_map is None:
        rename_map = ALL_RENAME_MAP
    df = df.copy()
    df[col] = df[col].map(lambda x: rename_map.get(x, x))
    df = df[df[col].notna()].reset_index(drop=True)
    return df


logger.info(f"Rename maps ready: {len(TOPIC_RENAME_MAP)} topic, {len(TRUTH_RENAME_MAP)} truth.\n")


## 3. Model Manager

The `ModelService` class centralizes all model loading and inference. Key design decisions:

- **Lazy LLM loading**: The LLM is not loaded until first use, keeping VRAM free during the embedding-heavy ingestion phase
- **Reranker scores are sigmoided**: Raw cross-encoder logits are passed through a sigmoid to normalize them to [0, 1], making them comparable across queries
- **Inference mode**: All encoding and scoring methods run under `torch.inference_mode()` to save memory

In [ ]:
class ModelService:
    def __init__(self, device: str = DEVICE):
        self.device = device

        logger.info("Loading embedding model...\n")
        self.embed_model = SentenceTransformer(
            EMBED_MODEL_ID,
            device=self.device,
            trust_remote_code=True,
        )
        self.embed_model.max_seq_length = 8192

        logger.info("Loading cross-encoder reranker...\n")
        from sentence_transformers import CrossEncoder
        self.cross_encoder = CrossEncoder(
            CROSS_ENCODER_MODEL_ID,
            device=self.device,
            trust_remote_code=True,
        )

        # LLM is lazy-loaded after ingestion to keep VRAM free for embeddings
        self._llm = None
        self.tokenizer = None

    @property
    def llm(self):
        if self._llm is None:
            self.load_llm()
        return self._llm

    @llm.setter
    def llm(self, value):
        self._llm = value

    @llm.deleter
    def llm(self):
        self._llm = None

    def load_llm(self):
        logger.info("Loading LLM tokenizer/model...\n")
        self.tokenizer = AutoTokenizer.from_pretrained(
            LLM_MODEL_ID,
            trust_remote_code=True,
        )
        if self.tokenizer.pad_token_id is None and self.tokenizer.eos_token_id is not None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self._llm = AutoModelForCausalLM.from_pretrained(
            LLM_MODEL_ID,
            trust_remote_code=True,
            device_map=self.device,
            dtype="auto",
            low_cpu_mem_usage=True,
        )
        logger.info("LLM loaded.\n")

        self._llm.eval()
        self.clear_cache()

    def clear_cache(self):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    @torch.inference_mode()
    def embed_texts(self, texts):
        emb = self.embed_model.encode(
            texts,
            batch_size=8,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        return emb

    @torch.inference_mode()
    def rerank_scores(self, query, docs):
        if not docs:
            return np.array([], dtype=np.float32)

        pairs = [[query, d] for d in docs]
        raw = self.cross_encoder.predict(
            pairs,
            batch_size=8,
            show_progress_bar=False,
        )
        scores = 1.0 / (1.0 + np.exp(-np.array(raw, dtype=np.float32)))
        return scores

service = ModelService(device=DEVICE)
logger.info(f"Reranker: using CrossEncoder ({CROSS_ENCODER_MODEL_ID})\n")


## 4. Data Ingestion

Documents are split into overlapping chunks using a recursive character splitter that tries natural boundaries (paragraphs, sentences, words) before falling back to fixed-width splits. Each chunk is embedded and stored in ChromaDB with its topic metadata.

The ingestion is incremental -- files already in the collection are skipped, so re-running this cell is safe and fast.

In [ ]:
def recursive_character_split(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP, seps=None):
    text = text.strip()
    if not text:
        return []

    if len(text) <= chunk_size:
        return [text]

    if seps is None or seps == []:
        seps = ["\n\n", "\n", ". ", " "]

    sep = seps[0]
    if sep in text:
        parts = text.split(sep)
        chunks, current = [], ""
        for part in parts:
            candidate = (current + sep + part).strip() if current else part.strip()
            if len(candidate) <= chunk_size:
                current = candidate
            else:
                if current:
                    chunks.append(current)
                if len(part) > chunk_size and len(seps) > 1:
                    chunks.extend(recursive_character_split(part, chunk_size, overlap, seps[1:]))
                    current = ""
                else:
                    current = part
        if current:
            chunks.append(current)
    else:
        step = max(1, chunk_size - overlap)
        chunks = [text[i:i + chunk_size] for i in range(0, len(text), step)]

    if overlap > 0 and len(chunks) > 1:
        merged = [chunks[0]]
        for i in range(1, len(chunks)):
            prev_tail = merged[-1][-overlap:] if len(merged[-1]) > overlap else merged[-1]
            merged.append((prev_tail + " " + chunks[i]).strip()[:chunk_size])
        chunks = merged

    return [c for c in chunks if c]


def _normalize_topic_name(name: str):
    cleaned = re.sub(r"\s+", " ", name).strip().lower()
    cleaned = cleaned.replace("_", " ")
    return cleaned


INGEST_DATA_PATH = Path(DATA_PATH)

if INGEST_DATA_PATH.exists():
    logger.info(f"Using local ingestion path: {INGEST_DATA_PATH}\n")
else:
    raise FileNotFoundError(f"Local data path not found: {INGEST_DATA_PATH}")

topics_json_path = INGEST_DATA_PATH / "topics.json"
if not topics_json_path.exists():
    raise FileNotFoundError(f"topics.json not found at: {topics_json_path}")

with open(topics_json_path, "r", encoding="utf-8") as f:
    topic_name_to_id = json.load(f)

normalized_topic_to_id = {
    _normalize_topic_name(k): int(v)
    for k, v in topic_name_to_id.items()
}

topic_id_to_name = {int(v): k for k, v in topic_name_to_id.items()}


def parse_metadata_from_path(file_path: Path):
    topic_id = -1
    topic_name = None

    for part in reversed(file_path.parts):
        norm = _normalize_topic_name(part)
        if norm in normalized_topic_to_id:
            topic_id = normalized_topic_to_id[norm]
            topic_name = topic_id_to_name[topic_id]
            break

    return topic_id, topic_name


client = chromadb.PersistentClient(
    path=DB_PATH,
    settings=chromadb.Settings(anonymized_telemetry=False),
)

FORCE_REBUILD_COLLECTION = False

if FORCE_REBUILD_COLLECTION:
    try:
        client.delete_collection(COLLECTION_NAME)
        logger.info(f"Deleted existing collection: {COLLECTION_NAME}\n")
    except Exception:
        pass

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

kb_root = INGEST_DATA_PATH / "topics"
if not kb_root.exists():
    kb_root = INGEST_DATA_PATH

text_files = sorted(list(kb_root.rglob("*.txt")) + list(kb_root.rglob("*.md")))

existing_count = collection.count()
already_ingested = set()
if existing_count > 0 and not FORCE_REBUILD_COLLECTION:
    all_ids = collection.get(include=[])["ids"]
    already_ingested = {id_.rsplit("::", 1)[0] for id_ in all_ids}

pending_files = [fp for fp in text_files if fp.as_posix() not in already_ingested]

if not text_files:
    logger.warning(f"No source files found under {kb_root}. Skipping ingestion.\n")
elif not pending_files:
    logger.info(
        f"All {len(text_files)} files already ingested ({existing_count} chunks). Nothing to do.\n"
    )
else:
    logger.info(f"Found {len(pending_files)}/{len(text_files)} files to ingest ({len(text_files) - len(pending_files)} already done).\n")

    batch_size = 64
    buffer_texts, buffer_ids, buffer_meta = [], [], []
    skipped_no_topic = 0

    def flush_ingest_buffers():
        if not buffer_texts:
            return 0
        b_emb = service.embed_texts(buffer_texts).astype(np.float32).tolist()
        collection.upsert(
            ids=buffer_ids,
            documents=buffer_texts,
            embeddings=b_emb,
            metadatas=buffer_meta,
        )
        n = len(buffer_texts)
        buffer_texts.clear()
        buffer_ids.clear()
        buffer_meta.clear()
        service.clear_cache()
        return n

    upserted = 0
    for fp in tqdm(pending_files, desc="Ingesting"):
        try:
            raw = fp.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            raw = fp.read_text(errors="ignore")

        topic_id, topic_name = parse_metadata_from_path(fp)
        if topic_id < 0:
            skipped_no_topic += 1
            continue

        chunks = recursive_character_split(raw, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
        for idx, chunk in enumerate(chunks):
            buffer_ids.append(f"{fp.as_posix()}::{idx}")
            buffer_texts.append(chunk)
            meta = {
                "source": fp.as_posix(),
                "topic_id": int(topic_id),
                "topic_name": topic_name,
                "chunk_id": idx,
            }
            buffer_meta.append(meta)

            if len(buffer_texts) >= batch_size:
                upserted += flush_ingest_buffers()

    upserted += flush_ingest_buffers()
    logger.info(
        f"Ingestion complete. Upserted chunks: {upserted} | Collection count: {collection.count()} | Skipped(no topic): {skipped_no_topic}\n"
    )


## 5. Retrieval-Based Topic Identification

Before involving the LLM, we benchmark several pure-retrieval strategies for topic identification. The idea is simple: retrieve the most relevant chunks for a query, then infer the topic from their metadata.

We compare three families of approaches:
- **Embedding-only**: Use cosine similarity to find the nearest chunks. We test top-1 (take the single best match) and weighted voting across k=5, 12, or 25 neighbors
- **Rerank top-1**: Retrieve k candidates by embedding, rerank with the cross-encoder, and take the topic of the highest-scoring chunk
- **Rerank sum-aggregate**: After reranking, sum scores per topic -- more robust when multiple chunks from the correct topic appear in the results

These retrieval-only strategies are fast (no LLM call) and serve as strong baselines.

In [ ]:
def _retrieve(query, k=25):
    """Embed the query and return the top-k chunks from ChromaDB."""
    q_emb = service.embed_texts([query])[0].astype(np.float32).tolist()
    out = collection.query(
        query_embeddings=[q_emb],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    docs  = out.get("documents", [[]])[0]
    metas = out.get("metadatas", [[]])[0]
    dists = out.get("distances", [[]])[0]
    return docs, metas, dists


def _safe_topic(meta):
    return int((meta or {}).get("topic_id", -1))


def _topic_vote_from_distances(metas, dists):
    """Sum (1-distance) per topic; return the topic with the highest sum."""
    topic_scores = {}
    best_doc_idx = {}
    for i, (meta, dist) in enumerate(zip(metas, dists)):
        tid = _safe_topic(meta)
        if tid < 0:
            continue
        score = 1.0 - float(dist)
        topic_scores[tid] = topic_scores.get(tid, 0.0) + score
        if tid not in best_doc_idx or score > (1.0 - float(dists[best_doc_idx[tid]])):
            best_doc_idx[tid] = i
    if not topic_scores:
        return -1, None, topic_scores
    best = max(topic_scores, key=topic_scores.get)
    return best, best_doc_idx.get(best), topic_scores


def _topic_from_rerank(metas, rerank_scores, mode="max"):
    """Aggregate reranker scores per topic (max or sum)."""
    topic_scores = {}
    best_doc_idx = {}
    for i, (meta, sc) in enumerate(zip(metas, rerank_scores)):
        tid = _safe_topic(meta)
        if tid < 0:
            continue
        s = float(sc)
        if mode == "sum":
            topic_scores[tid] = topic_scores.get(tid, 0.0) + s
        else:
            topic_scores[tid] = max(topic_scores.get(tid, -1e9), s)
        if tid not in best_doc_idx or s > float(rerank_scores[best_doc_idx[tid]]):
            best_doc_idx[tid] = i
    if not topic_scores:
        return -1, None, topic_scores
    best = max(topic_scores, key=topic_scores.get)
    return best, best_doc_idx.get(best), topic_scores


def topic_embedding_top1(query):
    docs, metas, _ = _retrieve(query, k=1)
    if not metas:
        return {"topic_id": -1}
    return {"topic_id": _safe_topic(metas[0])}


def _topic_embedding_vote(query, k):
    docs, metas, dists = _retrieve(query, k=k)
    tid, _, _ = _topic_vote_from_distances(metas, dists)
    return {"topic_id": tid}

def topic_embedding_vote_k5(query):  return _topic_embedding_vote(query, k=5)
def topic_embedding_vote_k12(query): return _topic_embedding_vote(query, k=12)
def topic_embedding_vote_k25(query): return _topic_embedding_vote(query, k=25)


def topic_rerank_top1_k12(query):
    docs, metas, _ = _retrieve(query, k=12)
    if not docs:
        return {"topic_id": -1}
    scores = service.rerank_scores(query, docs)
    tid, _, _ = _topic_from_rerank(metas, scores, mode="max")
    return {"topic_id": tid}

def topic_rerank_sum_k12(query):
    docs, metas, _ = _retrieve(query, k=12)
    if not docs:
        return {"topic_id": -1}
    scores = service.rerank_scores(query, docs)
    tid, _, _ = _topic_from_rerank(metas, scores, mode="sum")
    return {"topic_id": tid}

def topic_rerank_top1_k5(query):
    docs, metas, _ = _retrieve(query, k=5)
    if not docs:
        return {"topic_id": -1}
    scores = service.rerank_scores(query, docs)
    tid, _, _ = _topic_from_rerank(metas, scores, mode="max")
    return {"topic_id": tid}

def topic_rerank_top1_k25(query):
    docs, metas, _ = _retrieve(query, k=25)
    if not docs:
        return {"topic_id": -1}
    scores = service.rerank_scores(query, docs)
    tid, _, _ = _topic_from_rerank(metas, scores, mode="max")
    return {"topic_id": tid}


TOPIC_STRATEGIES = [
    ("embed_top1",         topic_embedding_top1),
    ("embed_vote_k5",      topic_embedding_vote_k5),
    ("embed_vote_k12",     topic_embedding_vote_k12),
    ("embed_vote_k25",     topic_embedding_vote_k25),
    ("rerank_max_k5",      topic_rerank_top1_k5),
    ("rerank_max_k12",     topic_rerank_top1_k12),
    ("rerank_max_k25",     topic_rerank_top1_k25),
    ("rerank_sum_k12",     topic_rerank_sum_k12),
]

logger.info(f"Defined {len(TOPIC_STRATEGIES)} retrieval-based topic strategies.\n")


## 6. LLM-Based Strategies

Now we bring in the LLM for two tasks:

**Topic identification via source-chunk selection**: After retrieving and reranking, we pass the top-k chunks to the LLM with a structured prompt. The LLM must return JSON with both a true/false verdict and the ID of the chunk it relied on. We infer the topic from that chunk's metadata. This tests whether the LLM can pick the most relevant evidence better than simple score aggregation.

**Truth prediction**: Each strategy retrieves context (via embedding only, or embedding + reranking) and asks the LLM to fact-check the medical statement. We vary the retrieval method and the number of context chunks to find the best configuration. The prompt emphasizes that accuracy is critical in a medical setting and that the LLM must reason exclusively from the provided evidence.


> _____
> [The prompt we are using originates from Lasse Bærland Strands solution to the task during the NMiAI competition](https://github.com/Attention-Heads/emergency-healthcare-rag)
> ____


In [ ]:
import types

def build_system_prompt(docs):
    return f"""You are a meticulous medical fact-checking expert. Your task is to verify a medical statement based *only* on the provided context passages. Accuracy is critical in a medical setting, if the answer is wrong, someone may die.

**Instructions:**
1.  Read the <medical_statement> carefully to understand its core claim.
2.  Systematically review each numbered context chunk (`[CHUNK 0]`, `[CHUNK 1]`, etc.).
3.  Identify the **single chunk** that contains the most direct and definitive evidence to either prove or disprove the statement. This is the `source_chunk_id`.
4.  Based *exclusively* on the evidence within your chosen chunk, determine if the statement is true (1) or false (0).

Your output **must** be a single, valid JSON object containing the `statement_is_true` value and the `source_chunk_id` you used to make that determination. The `source_chunk_id` must be an integer between 0 and {len(docs) - 1}."""

def _llm_json_classify_with_source(self, query, docs):
    """Ask the LLM to fact-check a statement AND identify which chunk it relied on."""
    _ = self.llm  # trigger lazy-load so self.tokenizer is initialized
    if not docs:
        return {"is_true": 0, "source_chunk_id": -1, "raw": ""}

    context_parts = []
    for i, chunk_text in enumerate(docs):
        context_parts.append(f"[CHUNK {i} below]\n{chunk_text}\n[CHUNK {i} above]")
    context = "\n\n---\n\n".join(context_parts)

    system_prompt = build_system_prompt(docs)
    user_prompt = f"""Here is the information to perform your task.
<context_passages>
{context}
</context_passages>

<medical_statement>
{query}
</medical_statement>

Evaluate if the statement is true or false based on the context and identify the single most relevant source chunk ID. Your response must be a valid JSON object."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    prompt = self.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = self.tokenizer(prompt, return_tensors="pt").to(self.llm.device)
    generated = self.llm.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=self.tokenizer.pad_token_id,
        eos_token_id=self.tokenizer.eos_token_id,
        use_cache=True,
    )

    new_tokens = generated[0][inputs["input_ids"].shape[1]:]
    text = self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text,
                  flags=re.IGNORECASE | re.MULTILINE).strip()
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    candidate = match.group(0) if match else text

    try:
        parsed = json.loads(candidate)
        out = {
            "is_true": int(parsed.get("statement_is_true", parsed.get("is_true", 0))),
            "source_chunk_id": int(parsed.get("source_chunk_id", -1)),
            "raw": text,
        }
    except Exception as e:
        logger.warning(f"JSON parsing failed: {e} | Raw: {text[:240]}\n")
        out = {"is_true": 0, "source_chunk_id": -1, "raw": text}

    del inputs, generated, new_tokens
    self.clear_cache()
    return out


service.llm_json_classify = types.MethodType(_llm_json_classify_with_source, service)
logger.info("LLM prompt patched: returns {statement_is_true, source_chunk_id}.\n")


def _retrieve_rerank_llm(query, retrieve_k=12, top_k=3):
    docs, metas, _ = _retrieve(query, k=retrieve_k)
    if not docs:
        return {"is_true": 0, "source_chunk_id": -1, "raw": ""}, []

    scores = service.rerank_scores(query, docs)
    order  = np.argsort(scores)[::-1][:top_k]
    top_docs  = [docs[i]  for i in order]
    top_metas = [metas[i] for i in order]

    try:
        pred = service.llm_json_classify(query=query, docs=top_docs)
    except Exception as e:
        logger.warning(f"LLM inference failed: {e}\n")
        pred = {"is_true": 0, "source_chunk_id": -1, "raw": ""}

    return pred, top_metas


def _retrieve_embed_llm(query, k=3):
    docs, metas, _ = _retrieve(query, k=k)
    if not docs:
        return {"is_true": 0, "source_chunk_id": -1, "raw": ""}, []

    try:
        pred = service.llm_json_classify(query=query, docs=docs)
    except Exception as e:
        logger.warning(f"LLM inference failed: {e}\n")
        pred = {"is_true": 0, "source_chunk_id": -1, "raw": ""}

    return pred, metas


def topic_llm_rerank(query, retrieve_k=12, top_k=3):
    pred, top_metas = _retrieve_rerank_llm(query, retrieve_k, top_k)
    src_id = int(pred.get("source_chunk_id", -1))
    if 0 <= src_id < len(top_metas):
        tid = _safe_topic(top_metas[src_id])
    else:
        tid = -1
    return {"topic_id": tid, "is_true": pred.get("is_true", 0)}


TOPIC_STRATEGIES.append(("llm_rerank_k12_top3", lambda q: topic_llm_rerank(q, 12, 3)))
logger.info(f"Total topic strategies: {len(TOPIC_STRATEGIES)}\n")


def truth_embed_top3(query):
    pred, _ = _retrieve_embed_llm(query, k=3)
    return {"is_true": pred.get("is_true", 0)}

def truth_embed_top5(query):
    pred, _ = _retrieve_embed_llm(query, k=5)
    return {"is_true": pred.get("is_true", 0)}

def truth_rerank_top3(query):
    pred, _ = _retrieve_rerank_llm(query, retrieve_k=12, top_k=3)
    return {"is_true": pred.get("is_true", 0)}

def truth_rerank_top5(query):
    pred, _ = _retrieve_rerank_llm(query, retrieve_k=12, top_k=5)
    return {"is_true": pred.get("is_true", 0)}

def truth_rerank_top3_k25(query):
    pred, _ = _retrieve_rerank_llm(query, retrieve_k=25, top_k=3)
    return {"is_true": pred.get("is_true", 0)}


TRUTH_STRATEGIES = [
    ("truth_embed_top3",       truth_embed_top3),
    ("truth_embed_top5",       truth_embed_top5),
    ("truth_rerank_top3",      truth_rerank_top3),
    ("truth_rerank_top5",      truth_rerank_top5),
    ("truth_rerank_top3_k25",  truth_rerank_top3_k25),
]

logger.info(f"Defined {len(TRUTH_STRATEGIES)} truth-prediction strategies.\n")


## 7. Benchmark Execution

We run every strategy against the ground-truth dataset in two passes:

- **Part A (Topic)**: Each of the 9 topic strategies predicts the topic for every statement. We record the prediction, correctness, and latency.
- **Part B (Truth)**: Each of the 5 truth strategies predicts true/false for every statement.

Results are **checkpointed to disk after every single prediction**. If the notebook crashes or is interrupted at any point, simply re-run this cell -- it will skip everything already saved and resume from where it left off.

In [ ]:
def load_ground_truth(data_path=DATA_PATH):
    statements_dir = Path(data_path) / "train" / "statements"
    answers_dir    = Path(data_path) / "train" / "answers"

    if not statements_dir.exists() or not answers_dir.exists():
        logger.warning(f"Train folders not found at {statements_dir} / {answers_dir}\n")
        return pd.DataFrame(columns=["statement_id", "query", "topic_id", "is_true"]), 0

    rows, missing = [], 0
    for stmt_path in sorted(statements_dir.glob("statement_*.txt")):
        sid = stmt_path.stem
        ans_path = answers_dir / f"{sid}.json"
        if not ans_path.exists():
            missing += 1
            continue
        query = stmt_path.read_text(encoding="utf-8", errors="ignore").strip()
        ans   = json.loads(ans_path.read_text(encoding="utf-8"))
        rows.append({
            "statement_id": sid,
            "query":        query,
            "topic_id":     int(ans.get("statement_topic", -1)),
            "is_true":      int(ans.get("statement_is_true", 0)),
        })

    df = pd.DataFrame(rows)
    logger.info(f"Ground truth loaded: {len(df)} statements | {missing} missing answers\n")
    return df, missing


gt_df, missing_answers = load_ground_truth(DATA_PATH)

# ── Part A: Topic Accuracy Benchmark ────────────────────────────────────────
topic_ckpt = Checkpoint(TOPIC_CKPT, key_fields=["statement_id", "strategy"])
logger.info(
    f"Topic benchmark: {len(TOPIC_STRATEGIES)} strategies x {len(gt_df)} statements. "
    f"Already done: {topic_ckpt.count()} rows.\n"
)

for _, row in tqdm(gt_df.iterrows(), total=len(gt_df), desc="Topic Benchmark"):
    query    = str(row["query"])
    gt_topic = int(row["topic_id"])
    sid      = str(row["statement_id"])

    for strat_name, strat_fn in TOPIC_STRATEGIES:
        if topic_ckpt.done(statement_id=sid, strategy=strat_name):
            continue  # already saved, skip

        t0 = time.perf_counter()
        try:
            pred = strat_fn(query)
        except Exception as e:
            logger.warning(f"[{strat_name}] {sid} failed: {e}\n")
            pred = {"topic_id": -1}
        latency = (time.perf_counter() - t0) * 1000.0

        pred_topic = int(pred.get("topic_id", -1))
        topic_ckpt.write({
            "statement_id":  sid,
            "strategy":      strat_name,
            "gt_topic_id":   gt_topic,
            "pred_topic_id": pred_topic,
            "is_correct":    pred_topic == gt_topic,
            "latency_ms":    round(latency, 3),
            "pred_is_true":  int(pred.get("is_true", -1)),
        })

    service.clear_cache()

topic_results_df = apply_renames(topic_ckpt.to_dataframe(), TOPIC_RENAME_MAP)
logger.info(f"Topic benchmark complete: {len(topic_results_df)} rows total.\n")
display(topic_results_df.head(10))


# ── Part B: Truth Accuracy Benchmark ────────────────────────────────────────
truth_ckpt = Checkpoint(TRUTH_CKPT, key_fields=["statement_id", "strategy"])
logger.info(
    f"Truth benchmark: {len(TRUTH_STRATEGIES)} strategies x {len(gt_df)} statements. "
    f"Already done: {truth_ckpt.count()} rows.\n"
)

for _, row in tqdm(gt_df.iterrows(), total=len(gt_df), desc="Truth Benchmark"):
    query    = str(row["query"])
    gt_truth = int(row["is_true"])
    gt_topic = int(row["topic_id"])
    sid      = str(row["statement_id"])

    for strat_name, strat_fn in TRUTH_STRATEGIES:
        if truth_ckpt.done(statement_id=sid, strategy=strat_name):
            continue  # already saved, skip

        t0 = time.perf_counter()
        try:
            pred = strat_fn(query)
        except Exception as e:
            logger.warning(f"[{strat_name}] {sid} failed: {e}\n")
            pred = {"is_true": 0}
        latency = (time.perf_counter() - t0) * 1000.0

        pred_truth = int(pred.get("is_true", 0))
        truth_ckpt.write({
            "statement_id":  sid,
            "strategy":      strat_name,
            "gt_topic_id":   gt_topic,
            "gt_is_true":    gt_truth,
            "pred_is_true":  pred_truth,
            "is_correct":    pred_truth == gt_truth,
            "latency_ms":    round(latency, 3),
            "raw_pred":      str(pred.get("raw", "")),
        })

    service.clear_cache()

truth_results_df = apply_renames(truth_ckpt.to_dataframe(), TRUTH_RENAME_MAP)
logger.info(f"Truth benchmark complete: {len(truth_results_df)} rows total.\n")
display(truth_results_df.head(10))


## 8. Topic Accuracy Analysis

We analyze topic identification performance across all strategies. The visualizations include:

1. **Accuracy table** sorted by performance, showing invalid-prediction rates and latency
2. **Horizontal bar charts** comparing accuracy and latency, color-coded by strategy family (embedding-only, reranked, LLM-based)
3. **Confusion matrix** for the best strategy, revealing which topics are most commonly confused

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

if topic_results_df.empty:
    print("No topic benchmark results. Check data paths.")
else:
    topic_metric_rows = []
    for strat, grp in topic_results_df.groupby("strategy"):
        acc = grp["is_correct"].mean() * 100
        invalid_rate = (grp["pred_topic_id"] < 0).mean() * 100
        topic_metric_rows.append({
            "strategy":         strat,
            "topic_acc_%":      round(acc, 2),
            "invalid_topic_%":  round(invalid_rate, 2),
            "avg_latency_ms":   round(grp["latency_ms"].mean(), 1),
        })

    topic_metrics_df = pd.DataFrame(topic_metric_rows).sort_values(
        "topic_acc_%", ascending=False
    )
    print("=" * 60)
    print("  TOPIC ACCURACY -- Strategy Comparison")
    print("=" * 60)
    display(topic_metrics_df)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    colors = []
    for s in topic_metrics_df["strategy"]:
        if "llm" in s:
            colors.append("firebrick")
        elif "rerank" in s:
            colors.append("darkorange")
        else:
            colors.append("steelblue")

    axes[0].barh(topic_metrics_df["strategy"], topic_metrics_df["topic_acc_%"], color=colors)
    axes[0].set_xlabel("Topic Accuracy (%)")
    axes[0].set_title("Topic Accuracy by Strategy")
    axes[0].set_xlim(0, 105)
    axes[0].invert_yaxis()

    axes[1].barh(topic_metrics_df["strategy"], topic_metrics_df["avg_latency_ms"], color=colors)
    axes[1].set_xlabel("Avg Latency (ms)")
    axes[1].set_title("Latency by Strategy")
    axes[1].invert_yaxis()

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor="steelblue",  label="Embedding only"),
        Patch(facecolor="darkorange", label="Reranked"),
        Patch(facecolor="firebrick",  label="LLM-based"),
    ]
    axes[0].legend(handles=legend_elements, loc="lower right")
    plt.tight_layout()
    plt.show()

    best_strat = topic_metrics_df.iloc[0]["strategy"]
    best_grp   = topic_results_df[topic_results_df["strategy"] == best_strat]
    labels     = sorted(best_grp["gt_topic_id"].unique())
    cm         = confusion_matrix(
        best_grp["gt_topic_id"], best_grp["pred_topic_id"], labels=labels
    )
    label_names = [topic_id_to_name.get(l, str(l)) for l in labels]

    fig2, ax2 = plt.subplots(
        figsize=(max(8, len(labels)), max(6, int(len(labels) * 0.7)))
    )
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=label_names, yticklabels=label_names, ax=ax2,
    )
    ax2.set_title(f"Topic Confusion Matrix -- {best_strat}")
    ax2.set_xlabel("Predicted Topic")
    ax2.set_ylabel("True Topic")
    plt.xticks(rotation=35, ha="right", fontsize=7)
    plt.yticks(rotation=0, fontsize=7)
    plt.tight_layout()
    plt.show()
    print(f"Confusion matrix shown for best strategy: {best_strat}")


## 9. Truth Prediction Analysis

For truth prediction we go beyond accuracy and report precision, recall, F1-score, and AUROC. In a medical fact-checking context, these metrics matter because:

- **Precision**: How often a "true" prediction is actually correct (avoiding false endorsements)
- **Recall**: How many truly true statements we correctly identify (avoiding false rejections)
- **F1**: The harmonic mean balancing both concerns

The precision-recall scatter plot helps us understand the trade-off each strategy makes.

In [ ]:
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
)
from matplotlib.patches import Patch

if truth_results_df.empty:
    print("No truth benchmark results. Check data paths.")
else:
    truth_metric_rows = []
    for strat, grp in truth_results_df.groupby("strategy"):
        y_true = grp["gt_is_true"].values
        y_pred = grp["pred_is_true"].values
        try:
            auroc = roc_auc_score(y_true, y_pred)
        except Exception:
            auroc = float("nan")
        truth_metric_rows.append({
            "strategy":       strat,
            "truth_acc_%":    round((y_true == y_pred).mean() * 100, 2),
            "precision":      round(precision_score(y_true, y_pred, zero_division=0), 4),
            "recall":         round(recall_score(y_true, y_pred, zero_division=0), 4),
            "f1":             round(f1_score(y_true, y_pred, zero_division=0), 4),
            "auroc":          round(auroc, 4),
            "avg_latency_ms": round(grp["latency_ms"].mean(), 1),
        })

    truth_metrics_df = pd.DataFrame(truth_metric_rows).sort_values(
        "f1", ascending=False
    )
    print("=" * 60)
    print("  TRUTH ACCURACY -- Strategy Comparison")
    print("=" * 60)
    display(truth_metrics_df)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    colors = []
    for s in truth_metrics_df["strategy"]:
        if "rerank" in s:
            colors.append("darkorange")
        else:
            colors.append("steelblue")

    for ax, col, title, ylim in zip(
        axes,
        ["truth_acc_%", "f1", "avg_latency_ms"],
        ["Truth Accuracy (%)", "F1-Score", "Avg Latency (ms)"],
        [(0, 105), (0, 1.05), None],
    ):
        ax.barh(truth_metrics_df["strategy"], truth_metrics_df[col], color=colors)
        ax.set_xlabel(title)
        ax.set_title(title)
        if ylim:
            ax.set_xlim(*ylim)
        ax.invert_yaxis()

    legend_elements = [
        Patch(facecolor="steelblue",  label="Embedding + LLM"),
        Patch(facecolor="darkorange", label="Reranked + LLM"),
    ]
    axes[0].legend(handles=legend_elements, loc="lower right")
    plt.tight_layout()
    plt.show()

    fig3, ax3 = plt.subplots(figsize=(7, 5))
    for _, row in truth_metrics_df.iterrows():
        ax3.scatter(row["recall"], row["precision"], s=100, zorder=5)
        ax3.annotate(
            row["strategy"],
            (row["recall"], row["precision"]),
            textcoords="offset points", xytext=(5, 5), fontsize=7,
        )
    ax3.set_xlabel("Recall")
    ax3.set_ylabel("Precision")
    ax3.set_title("Precision vs Recall -- Truth Prediction Strategies")
    ax3.set_xlim(0, 1.05)
    ax3.set_ylim(0, 1.05)
    ax3.plot([0, 1], [0, 1], "--", color="grey", alpha=0.4)
    plt.tight_layout()
    plt.show()


## 10. Token-Length Profiling

Before running the chunking ablation study, we need to understand how our documents tokenize under the embedding model's tokenizer. This analysis answers key questions:

- How long are our source documents in tokens?
- At each candidate chunk size, what is the actual token-length distribution?
- Do any chunks exceed the embedding model's `max_seq_length` (which would cause silent truncation)?

The histograms show a red line at the model's maximum sequence length. Any chunks to the right of that line would lose information during embedding.

In [ ]:
from transformers import AutoTokenizer as _AT
_tok = _AT.from_pretrained(EMBED_MODEL_ID, trust_remote_code=True)

kb_root = INGEST_DATA_PATH / "topics"
if not kb_root.exists():
    kb_root = INGEST_DATA_PATH
text_files = sorted(list(kb_root.rglob("*.txt")) + list(kb_root.rglob("*.md")))

doc_token_lens = []
for fp in text_files:
    try:
        raw = fp.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        raw = fp.read_text(errors="ignore")
    doc_token_lens.append(len(_tok.encode(raw, add_special_tokens=False)))

doc_lens = np.array(doc_token_lens)

print("=== Full-Document Token Lengths ===")
print(f"  Documents:  {len(doc_lens)}")
print(f"  Min:        {doc_lens.min()}")
print(f"  Median:     {int(np.median(doc_lens))}")
print(f"  Mean:       {doc_lens.mean():.0f}")
print(f"  Max:        {doc_lens.max()}")
print(f"  Std:        {doc_lens.std():.0f}")
for pct in [75, 90, 95, 99]:
    print(f"  P{pct}:       {int(np.percentile(doc_lens, pct))}")

print()
print("=== Chunk Token Lengths per chunk_size ===")
chunk_sizes_to_check = [256, 512, 1024, 2048, 4096, 8192, 16384]
chunk_stats_rows = []

for cs in chunk_sizes_to_check:
    overlap = cs // 4
    chunk_tok_lens = []
    for fp in text_files:
        try:
            raw = fp.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            raw = fp.read_text(errors="ignore")
        chunks = recursive_character_split(raw, chunk_size=cs, overlap=overlap)
        for ch in chunks:
            chunk_tok_lens.append(len(_tok.encode(ch, add_special_tokens=False)))

    arr = np.array(chunk_tok_lens)
    row = {
        "chunk_size (chars)": cs,
        "overlap": overlap,
        "num_chunks": len(arr),
        "tok_min": arr.min(),
        "tok_median": int(np.median(arr)),
        "tok_mean": f"{arr.mean():.0f}",
        "tok_max": arr.max(),
        "tok_p95": int(np.percentile(arr, 95)),
    }
    chunk_stats_rows.append(row)

chunk_stats_df = pd.DataFrame(chunk_stats_rows)
display(chunk_stats_df)

n = len(chunk_sizes_to_check)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 3))
for ax, cs in zip(axes, chunk_sizes_to_check):
    overlap = cs // 4
    tok_lens = []
    for fp in text_files:
        try:
            raw = fp.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            raw = fp.read_text(errors="ignore")
        for ch in recursive_character_split(raw, chunk_size=cs, overlap=overlap):
            tok_lens.append(len(_tok.encode(ch, add_special_tokens=False)))
    ax.hist(tok_lens, bins=30, color="teal", edgecolor="white", alpha=0.8)
    ax.set_title(f"chunk_size={cs}")
    ax.set_xlabel("Token length")
    ax.set_ylabel("Count")
    ax.axvline(x=service.embed_model.max_seq_length, color="red", linestyle="--",
               label=f"max_seq_len={service.embed_model.max_seq_length}")
    ax.legend(fontsize=7)

plt.suptitle("Chunk Token-Length Distributions", fontsize=12)
plt.tight_layout()
plt.show()

del _tok


## 11. Ablation Study: Chunking Parameters

Armed with the token-length analysis, we now sweep over different chunk sizes and overlaps to find the empirical optimum. For each configuration we:

1. Re-chunk and re-ingest the entire corpus into a temporary ChromaDB collection
2. Run the best retrieval-only strategy (identified in Section 8) against the ground truth
3. Record topic accuracy and chunk count

Each individual prediction is **checkpointed immediately**. If interrupted, re-running this cell will skip already-completed predictions and resume.

In [ ]:
ABLATION_MAX_QUERIES = 50   # set to None for full benchmark set
ABLATION_CONFIGS = [
    (256,    32),
    (512,    64),
    (512,   128),
    (1024,  128),
    (1024,  256),
    (2048,  256),
    (2048,  512),
    (4096,  1024),
    (8192,  2048),
    (16384, 4096),
]

# Identify the best retrieval-only topic strategy to use for ablation
_retrieval_only = topic_metrics_df[~topic_metrics_df["strategy"].str.contains("llm")]
best_retrieval_name = _retrieval_only.iloc[0]["strategy"]
best_retrieval_fn   = dict([(TOPIC_RENAME_MAP.get(k, k), v) for k, v in TOPIC_STRATEGIES])[best_retrieval_name]
logger.info(f"Ablation uses best retrieval strategy: '{best_retrieval_name}'\n")

ablation_gt = gt_df.head(ABLATION_MAX_QUERIES) if ABLATION_MAX_QUERIES else gt_df

# One checkpoint for all configs -- keyed by (chunk_size, chunk_overlap, statement_id)
abl_ckpt = Checkpoint(ABLATION_CKPT, key_fields=["chunk_size", "chunk_overlap", "statement_id"])
logger.info(f"Ablation checkpoint: already done {abl_ckpt.count()} individual predictions.\n")

for chunk_size, chunk_overlap in ABLATION_CONFIGS:
    col_name = f"ablation_{chunk_size}_{chunk_overlap}"
    logger.info(f"Ablation: chunk_size={chunk_size}, overlap={chunk_overlap}\n")

    abl_col = client.get_or_create_collection(
        name=col_name, metadata={"hnsw:space": "cosine"}
    )

    if abl_col.count() > 0:
        logger.info(f"  '{col_name}' already has {abl_col.count()} chunks, skipping ingest.\n")
    else:
        kb_root = INGEST_DATA_PATH / "topics"
        if not kb_root.exists():
            kb_root = INGEST_DATA_PATH
        text_files = sorted(list(kb_root.rglob("*.txt")) + list(kb_root.rglob("*.md")))

        abl_texts, abl_ids, abl_metas = [], [], []

        def _flush_abl():
            if not abl_texts:
                return
            embs = service.embed_texts(abl_texts).astype(np.float32).tolist()
            abl_col.upsert(
                ids=abl_ids, documents=abl_texts, embeddings=embs, metadatas=abl_metas,
            )
            abl_texts.clear(); abl_ids.clear(); abl_metas.clear()
            service.clear_cache()

        for fp in tqdm(text_files, desc=f"Ingest {chunk_size}/{chunk_overlap}"):
            try:
                raw = fp.read_text(encoding="utf-8", errors="ignore")
            except Exception:
                raw = fp.read_text(errors="ignore")
            topic_id, topic_name = parse_metadata_from_path(fp)
            if topic_id < 0:
                continue
            for idx, chunk in enumerate(
                recursive_character_split(raw, chunk_size, chunk_overlap)
            ):
                abl_ids.append(f"{fp.as_posix()}::{idx}")
                abl_texts.append(chunk)
                abl_metas.append({
                    "source": fp.as_posix(),
                    "topic_id": int(topic_id),
                    "topic_name": topic_name,
                    "chunk_id": idx,
                })
                if len(abl_texts) >= 64:
                    _flush_abl()
        _flush_abl()

    # --- run predictions, checkpointing each one ---
    _orig_collection = collection
    try:
        globals()["collection"] = abl_col

        for _, row in tqdm(
            ablation_gt.iterrows(),
            total=len(ablation_gt),
            desc=f"Ablation {chunk_size}/{chunk_overlap}",
        ):
            sid = str(row["statement_id"])
            if abl_ckpt.done(chunk_size=chunk_size, chunk_overlap=chunk_overlap, statement_id=sid):
                continue

            t0 = time.perf_counter()
            try:
                pred = best_retrieval_fn(str(row["query"]))
            except Exception as e:
                logger.warning(f"Ablation {chunk_size}/{chunk_overlap} {sid}: {e}\n")
                pred = {"topic_id": -1}
            latency = (time.perf_counter() - t0) * 1000.0

            pred_topic = int(pred.get("topic_id", -1))
            abl_ckpt.write({
                "chunk_size":         chunk_size,
                "chunk_overlap":      chunk_overlap,
                "num_chunks":         abl_col.count(),
                "statement_id":       sid,
                "gt_topic_id":        int(row["topic_id"]),
                "pred_topic_id":      pred_topic,
                "is_correct":         pred_topic == int(row["topic_id"]),
                "latency_ms":         round(latency, 3),
                "retrieval_strategy": best_retrieval_name,
            })

    finally:
        globals()["collection"] = _orig_collection

    logger.info(
        f"  Config {chunk_size}/{chunk_overlap} done. "
        f"Total checkpoint rows: {abl_ckpt.count()}\n"
    )

# Aggregate for display / plotting
abl_raw_df = abl_ckpt.to_dataframe()
abl_df = (
    abl_raw_df
    .groupby(["chunk_size", "chunk_overlap", "num_chunks"])
    .agg(
        topic_acc_pct=("is_correct", lambda x: round(x.mean() * 100, 2)),
        avg_latency_ms=("latency_ms", lambda x: round(x.mean(), 1)),
        n=("is_correct", "count"),
    )
    .reset_index()
    .rename(columns={"topic_acc_pct": "topic_acc_%"})
)
print("=" * 60)
print(f"  ABLATION STUDY -- Chunking Parameters ({best_retrieval_name})")
print("=" * 60)
display(abl_df)

# Bar chart
fig4, ax4 = plt.subplots(figsize=(10, 4))
labels_abl = [f"{r.chunk_size}/{r.chunk_overlap}" for r in abl_df.itertuples()]
bars = ax4.bar(labels_abl, abl_df["topic_acc_%"], color="teal")

best_idx = abl_df["topic_acc_%"].idxmax()
bars[best_idx].set_color("firebrick")

ax4.set_title(f"Topic Accuracy vs Chunk Size/Overlap -- {best_retrieval_name}")
ax4.set_xlabel("chunk_size / chunk_overlap")
ax4.set_ylabel("Topic Accuracy (%)")
ax4.set_ylim(0, 100)

for i_bar, v in enumerate(abl_df["topic_acc_%"]):
    ax4.text(i_bar, v + 1, f"{v:.1f}%", ha="center", fontsize=9)

plt.tight_layout()
plt.show()


## 12. LLM Model Comparison: Qwen3 vs Qwen3.5

Finally, we test whether a newer LLM improves performance. We swap out the current Qwen3-4B for Qwen3.5-4B and re-run all LLM-dependent strategies (topic identification via LLM and all truth prediction strategies).

The retrieval-only topic strategies are unaffected by the LLM swap, so we skip those and only re-run what matters. Each prediction is checkpointed immediately. Results are presented side-by-side with grouped bar charts for easy comparison.

In [ ]:
COMPARISON_LLM_ID = "Qwen/Qwen3.5-4B"

# --- 1. Pull baseline results from existing checkpoints ---
baseline_model_name = LLM_MODEL_ID.split("/")[-1]

baseline_topic_llm = apply_renames(topic_ckpt.to_dataframe(), TOPIC_RENAME_MAP)
baseline_topic_llm = baseline_topic_llm[
    baseline_topic_llm["strategy"] == "topic_llm_rerank_k12_top3"
].copy()
baseline_topic_llm["model"] = baseline_model_name

baseline_truth = apply_renames(truth_ckpt.to_dataframe(), TRUTH_RENAME_MAP).copy()
baseline_truth["model"] = baseline_model_name

# --- 2. Swap LLM to comparison model ---
logger.info(f"Unloading current LLM ({LLM_MODEL_ID})...\n")

# Move off GPU before deleting — critical step
if service._llm is not None:
    service._llm.cpu()
    del service._llm
    service._llm = None

if service.tokenizer is not None:
    del service.tokenizer
    service.tokenizer = None

gc.collect()
torch.cuda.synchronize()
torch.cuda.empty_cache()

# Verify it actually freed
allocated = torch.cuda.memory_allocated() / 1024**3
logger.info(f"VRAM after unload: {allocated:.2f} GB\n")

logger.info(f"Loading comparison LLM: {COMPARISON_LLM_ID}...\n")
service.tokenizer = AutoTokenizer.from_pretrained(
    COMPARISON_LLM_ID, trust_remote_code=True,
)
if service.tokenizer.pad_token_id is None and service.tokenizer.eos_token_id is not None:
    service.tokenizer.pad_token = service.tokenizer.eos_token

service.llm = AutoModelForCausalLM.from_pretrained(
    COMPARISON_LLM_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
).to(DEVICE)
service.llm.eval()
service.clear_cache()

import types
service.llm_json_classify = types.MethodType(_llm_json_classify_with_source, service)
logger.info(f"LLM swapped to {COMPARISON_LLM_ID}. Prompt re-patched.\n")

comparison_model_name = COMPARISON_LLM_ID.split("/")[-1]

# --- 3. Checkpointed LLM topic strategy ---
llm_topic_ckpt = Checkpoint(LLM_CMP_TOPIC_CKPT, key_fields=["model", "statement_id"])
logger.info(f"LLM topic comparison: already done {llm_topic_ckpt.count()} rows.\n")

for _, row in tqdm(gt_df.iterrows(), total=len(gt_df), desc=f"Topic LLM ({comparison_model_name})"):
    sid = str(row["statement_id"])
    if llm_topic_ckpt.done(model=comparison_model_name, statement_id=sid):
        continue

    t0 = time.perf_counter()
    try:
        pred = topic_llm_rerank(str(row["query"]), 12, 3)
    except Exception as e:
        logger.warning(f"{sid}: {e}\n")
        pred = {"topic_id": -1, "is_true": 0}
    latency = (time.perf_counter() - t0) * 1000.0

    pred_topic = int(pred.get("topic_id", -1))
    llm_topic_ckpt.write({
        "model":         comparison_model_name,
        "statement_id":  sid,
        "strategy":      "llm_rerank_k12_top3",
        "gt_topic_id":   int(row["topic_id"]),
        "pred_topic_id": pred_topic,
        "is_correct":    pred_topic == int(row["topic_id"]),
        "latency_ms":    round(latency, 3),
    })
    service.clear_cache()

# --- 4. Checkpointed truth strategies ---
llm_truth_ckpt = Checkpoint(LLM_CMP_TRUTH_CKPT, key_fields=["model", "statement_id", "strategy"])
logger.info(f"LLM truth comparison: already done {llm_truth_ckpt.count()} rows.\n")

for _, row in tqdm(gt_df.iterrows(), total=len(gt_df), desc=f"Truth ({comparison_model_name})"):
    sid      = str(row["statement_id"])
    gt_truth = int(row["is_true"])
    gt_topic = int(row["topic_id"])

    for strat_name, strat_fn in TRUTH_STRATEGIES:
        if llm_truth_ckpt.done(model=comparison_model_name, statement_id=sid, strategy=strat_name):
            continue

        t0 = time.perf_counter()
        try:
            pred = strat_fn(str(row["query"]))
        except Exception as e:
            logger.warning(f"{sid} {strat_name}: {e}\n")
            pred = {"is_true": 0}
        latency = (time.perf_counter() - t0) * 1000.0

        pred_truth = int(pred.get("is_true", 0))
        llm_truth_ckpt.write({
            "model":         comparison_model_name,
            "statement_id":  sid,
            "strategy":      strat_name,
            "gt_topic_id":   gt_topic,
            "gt_is_true":    gt_truth,
            "pred_is_true":  pred_truth,
            "is_correct":    pred_truth == gt_truth,
            "latency_ms":    round(latency, 3),
            "raw_pred":      str(pred.get("raw", "")),
        })
    service.clear_cache()

comp_topic_df = apply_renames(llm_topic_ckpt.to_dataframe(), TOPIC_RENAME_MAP)
comp_truth_df = apply_renames(llm_truth_ckpt.to_dataframe(), TRUTH_RENAME_MAP)

# --- 5. Combined analysis ---
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from matplotlib.patches import Patch

combined_topic = pd.concat([baseline_topic_llm, comp_topic_df], ignore_index=True)
topic_comp_rows = []
for model, grp in combined_topic.groupby("model"):
    acc = grp["is_correct"].mean() * 100
    invalid = (grp["pred_topic_id"] < 0).mean() * 100
    topic_comp_rows.append({
        "model": model,
        "topic_acc_%": round(acc, 2),
        "invalid_topic_%": round(invalid, 2),
        "avg_latency_ms": round(grp["latency_ms"].mean(), 1),
    })
topic_comp_df = pd.DataFrame(topic_comp_rows).sort_values("topic_acc_%", ascending=False)

print("=" * 60)
print("  LLM MODEL COMPARISON -- Topic Accuracy (topic_llm_rerank_k12_top3)")
print("=" * 60)
display(topic_comp_df)

combined_truth = pd.concat([baseline_truth, comp_truth_df], ignore_index=True)
truth_comp_rows = []
for (model, strat), grp in combined_truth.groupby(["model", "strategy"]):
    y_true = grp["gt_is_true"].values
    y_pred = grp["pred_is_true"].values
    try:
        auroc = roc_auc_score(y_true, y_pred)
    except Exception:
        auroc = float("nan")
    truth_comp_rows.append({
        "model":          model,
        "strategy":       strat,
        "truth_acc_%":    round((y_true == y_pred).mean() * 100, 2),
        "f1":             round(f1_score(y_true, y_pred, zero_division=0), 4),
        "precision":      round(precision_score(y_true, y_pred, zero_division=0), 4),
        "recall":         round(recall_score(y_true, y_pred, zero_division=0), 4),
        "auroc":          round(auroc, 4),
        "avg_latency_ms": round(grp["latency_ms"].mean(), 1),
    })

truth_comp_df = pd.DataFrame(truth_comp_rows).sort_values(
    ["strategy", "f1"], ascending=[True, False]
)
print("=" * 60)
print("  LLM MODEL COMPARISON -- Truth Accuracy")
print("=" * 60)
display(truth_comp_df)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

pivot_f1 = truth_comp_df.pivot(index="strategy", columns="model", values="f1")
pivot_f1.plot(kind="barh", ax=axes[0], color=["steelblue", "firebrick"])
axes[0].set_xlabel("F1-Score")
axes[0].set_title("Truth F1 by Strategy & Model")
axes[0].set_xlim(0, 1.05)
axes[0].legend(title="Model")

pivot_acc = truth_comp_df.pivot(index="strategy", columns="model", values="truth_acc_%")
pivot_acc.plot(kind="barh", ax=axes[1], color=["steelblue", "firebrick"])
axes[1].set_xlabel("Truth Accuracy (%)")
axes[1].set_title("Truth Accuracy by Strategy & Model")
axes[1].set_xlim(0, 105)
axes[1].legend(title="Model")

plt.tight_layout()
plt.show()

overall_rows = []
for model, grp in combined_truth.groupby("model"):
    y_true = grp["gt_is_true"].values
    y_pred = grp["pred_is_true"].values
    overall_rows.append({
        "model": model,
        "mean_truth_acc_%": round((y_true == y_pred).mean() * 100, 2),
        "mean_f1": round(f1_score(y_true, y_pred, zero_division=0), 4),
        "avg_latency_ms": round(grp["latency_ms"].mean(), 1),
    })
overall_df = pd.DataFrame(overall_rows).sort_values("mean_f1", ascending=False)
print("=" * 60)
print("  OVERALL MODEL SUMMARY (across all truth strategies)")
print("=" * 60)
display(overall_df)

# --- 6. Save everything to Parquet for easy analysis later ---
def save_parquet(df, name):
    path = RESULTS_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    logger.info(f"Saved {len(df)} rows -> {path}\n")

save_parquet(apply_renames(topic_ckpt.to_dataframe(), TOPIC_RENAME_MAP),     "topic_results")
save_parquet(apply_renames(truth_ckpt.to_dataframe(), TRUTH_RENAME_MAP),     "truth_results")
save_parquet(abl_ckpt.to_dataframe(),       "ablation_results")
save_parquet(apply_renames(llm_topic_ckpt.to_dataframe(), TOPIC_RENAME_MAP), "llm_cmp_topic_results")
save_parquet(apply_renames(llm_truth_ckpt.to_dataframe(), TRUTH_RENAME_MAP), "llm_cmp_truth_results")
logger.info("All results saved to Parquet. Done.\n")


## 12b. Error Analysis

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
 
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
 
# --- Topic strategies Pareto ---
ax = axes[0]
for _, row in topic_metrics_df.iterrows():
    s = row["strategy"]
    if "llm" in s:
        color, marker = "firebrick", "D"
    elif "rerank" in s:
        color, marker = "darkorange", "s"
    else:
        color, marker = "steelblue", "o"
    ax.scatter(
        row["avg_latency_ms"], row["topic_acc_%"],
        c=color, marker=marker, s=120, zorder=5, edgecolors="black", linewidths=0.5
    )
    ax.annotate(
        s, (row["avg_latency_ms"], row["topic_acc_%"]),
        textcoords="offset points", xytext=(6, 6), fontsize=7,
    )
 
ax.set_xscale("log")
ax.set_xlabel("Avg Latency (ms, log scale)")
ax.set_ylabel("Topic Accuracy (%)")
ax.set_title("Topic Identification: Accuracy vs Latency")
ax.set_ylim(70, 92)
 
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='steelblue', markersize=10, label='Embedding only'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='darkorange', markersize=10, label='Reranked'),
    Line2D([0], [0], marker='D', color='w', markerfacecolor='firebrick', markersize=10, label='LLM-based'),
]
ax.legend(handles=legend_elements, loc="lower right")
ax.grid(True, alpha=0.3)
 
# --- Truth strategies Pareto ---
ax = axes[1]
for _, row in truth_metrics_df.iterrows():
    s = row["strategy"]
    if "rerank" in s:
        color, marker = "darkorange", "s"
    else:
        color, marker = "steelblue", "o"
    ax.scatter(
        row["avg_latency_ms"], row["f1"],
        c=color, marker=marker, s=120, zorder=5, edgecolors="black", linewidths=0.5
    )
    ax.annotate(
        s, (row["avg_latency_ms"], row["f1"]),
        textcoords="offset points", xytext=(6, 6), fontsize=7,
    )
 
ax.set_xlabel("Avg Latency (ms)")
ax.set_ylabel("F1-Score")
ax.set_title("Truth Prediction: F1 vs Latency")
ax.set_ylim(0.7, 0.9)
 
legend_elements2 = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='steelblue', markersize=10, label='Embedding + LLM'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='darkorange', markersize=10, label='Reranked + LLM'),
]
ax.legend(handles=legend_elements2, loc="lower right")
ax.grid(True, alpha=0.3)
 
plt.suptitle("Pareto Analysis: Accuracy vs Latency Trade-off", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("results/pareto_accuracy_vs_latency.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: results/pareto_accuracy_vs_latency.png")

In [ ]:
import pandas as pd
 
best_strat = "topic_rerank_k25_top1"  # best topic strategy (renamed)
best_grp = topic_results_df[topic_results_df["strategy"] == best_strat].copy()
 
# Filter to only incorrect predictions
errors = best_grp[~best_grp["is_correct"]].copy()
errors["gt_topic_name"] = errors["gt_topic_id"].map(topic_id_to_name)
errors["pred_topic_name"] = errors["pred_topic_id"].map(topic_id_to_name)
 
# Count errors per ground-truth topic
topic_error_counts = (
    errors.groupby(["gt_topic_id", "gt_topic_name"])
    .size()
    .reset_index(name="num_errors")
    .sort_values("num_errors", ascending=False)
)
 
# Also count how many statements per topic total
topic_totals = (
    best_grp.groupby("gt_topic_id")
    .size()
    .reset_index(name="total_statements")
)
topic_error_counts = topic_error_counts.merge(topic_totals, on="gt_topic_id")
topic_error_counts["error_rate_%"] = (
    (topic_error_counts["num_errors"] / topic_error_counts["total_statements"] * 100).round(1)
)
 
print("=" * 70)
print(f"  TOP MISCLASSIFIED TOPICS ({best_strat})")
print("=" * 70)
top_n = 15
display(topic_error_counts.head(top_n))
 
# Most common confusion pairs
confusion_pairs = (
    errors.groupby(["gt_topic_name", "pred_topic_name"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
print("\n" + "=" * 70)
print("  MOST COMMON CONFUSION PAIRS")
print("=" * 70)
display(confusion_pairs.head(15))
 
# Bar chart of top misclassified topics
top_errors = topic_error_counts.head(10)
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(
    top_errors["gt_topic_name"],
    top_errors["num_errors"],
    color="coral", edgecolor="black", linewidth=0.5
)
ax.set_xlabel("Number of Misclassifications")
ax.set_title(f"Top 10 Most Misclassified Topics ({best_strat})")
ax.invert_yaxis()
 
# Annotate with error rate
for i, row in enumerate(top_errors.itertuples()):
    ax.text(row.num_errors + 0.1, i, f"{row.num_errors}/{row.total_statements}", va="center", fontsize=9)
 
plt.tight_layout()
plt.savefig("results/top_misclassified_topics.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: results/top_misclassified_topics.png")

In [ ]:
# Use the best truth strategy
best_truth_strat = "truth_rerank_k12_top5"
truth_best = truth_results_df[truth_results_df["strategy"] == best_truth_strat].copy()
 
# Merge with ground truth to get the actual statement text
truth_errors = truth_best[~truth_best["is_correct"]].copy()
truth_errors = truth_errors.merge(
    gt_df[["statement_id", "query", "topic_id"]],
    on="statement_id", how="left", suffixes=("", "_gt")
)
truth_errors["topic_name"] = truth_errors["gt_topic_id"].map(topic_id_to_name)
 
print("=" * 70)
print(f"  ERROR ANALYSIS -- {best_truth_strat}")
print(f"  Total errors: {len(truth_errors)} / {len(truth_best)}")
print("=" * 70)
 
# Show error breakdown: false positives vs false negatives
fp = truth_errors[truth_errors["pred_is_true"] == 1]  # predicted true, actually false
fn = truth_errors[truth_errors["pred_is_true"] == 0]  # predicted false, actually true
print(f"\n  False positives (predicted TRUE, actually FALSE): {len(fp)}")
print(f"  False negatives (predicted FALSE, actually TRUE):  {len(fn)}")
print(f"  -> The model is {'conservative (rejects too many true statements)' if len(fn) > len(fp) else 'permissive (accepts too many false statements)'}")
 
# Error distribution by topic
errors_by_topic = (
    truth_errors.groupby("topic_name")
    .size()
    .reset_index(name="errors")
    .sort_values("errors", ascending=False)
)
print("\n  Topics with most truth-prediction errors:")
display(errors_by_topic.head(10))
 
# Print example errors for manual inspection
print("\n" + "=" * 70)
print("  SAMPLE ERRORS (for report discussion section)")
print("=" * 70)
 
n_examples = min(8, len(truth_errors))
for i, (_, row) in enumerate(truth_errors.head(n_examples).iterrows()):
    gt_label = "TRUE" if row["gt_is_true"] == 1 else "FALSE"
    pred_label = "TRUE" if row["pred_is_true"] == 1 else "FALSE"
    error_type = "False Positive" if row["pred_is_true"] == 1 else "False Negative"
    print(f"\n--- Example {i+1} [{error_type}] ---")
    print(f"  Topic: {row.get('topic_name', 'N/A')}")
    print(f"  Ground truth: {gt_label} | Predicted: {pred_label}")
    print(f"  Statement: {row['query'][:300]}...")
    print()

In [ ]:
# Free the LLM — we only need embedder + reranker for this
if service._llm is not None:
    service._llm.cpu()
    del service._llm
    service._llm = None
if service.tokenizer is not None:
    del service.tokenizer
    service.tokenizer = None
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after unload: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

In [ ]:
print("=" * 70)
print("  DETAILED ERROR EXAMPLES WITH RETRIEVED CONTEXT")
print("=" * 70)
 
n_detailed = 3  # show full context for 3 errors
for i, (_, row) in enumerate(truth_errors.head(n_detailed).iterrows()):
    query = row["query"]
    gt_label = "TRUE" if row["gt_is_true"] == 1 else "FALSE"
    pred_label = "TRUE" if row["pred_is_true"] == 1 else "FALSE"
 
    # Retrieve context (same as truth_rerank_k12_top5 does)
    docs, metas, _ = _retrieve(query, k=12)
    if docs:
        scores = service.rerank_scores(query, docs)
        order = np.argsort(scores)[::-1][:5]
        top_docs = [docs[j] for j in order]
        top_metas = [metas[j] for j in order]
        top_scores = [float(scores[j]) for j in order]
    else:
        top_docs, top_metas, top_scores = [], [], []
 
    print(f"\n{'='*60}")
    print(f"ERROR EXAMPLE {i+1}")
    print(f"{'='*60}")
    print(f"Topic: {row.get('topic_name', 'N/A')}")
    print(f"Ground Truth: {gt_label} | Predicted: {pred_label}")
    print(f"Statement:\n  {query[:400]}")
    print(f"\nTop retrieved chunks (reranker scores):")
    for j, (doc, meta, sc) in enumerate(zip(top_docs, top_metas, top_scores)):
        chunk_topic = meta.get("topic_name", "?")
        print(f"  [{j}] score={sc:.4f} | topic={chunk_topic}")
        print(f"      {doc[:200]}...")
    print()

## 12c. Report Figures

Generate all figures for the report from checkpoint data. All figures are saved to `results/report_figures/`.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from pathlib import Path
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import jsonlines

# ── Config ──────────────────────────────────────────────────────────
RESULTS_DIR = Path("results")
FIG_DIR = RESULTS_DIR / "report_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

TOPIC_CKPT = RESULTS_DIR / "topic_results.jsonl"
TRUTH_CKPT = RESULTS_DIR / "truth_results.jsonl"
ABLATION_CKPT = RESULTS_DIR / "ablation_results.jsonl"
LLM_CMP_TOPIC_CKPT = RESULTS_DIR / "llm_cmp_topic_results.jsonl"
LLM_CMP_TRUTH_CKPT = RESULTS_DIR / "llm_cmp_truth_results.jsonl"

def load_jsonl(path):
    rows = []
    with jsonlines.open(path) as reader:
        for obj in reader:
            rows.append(obj)
    return pd.DataFrame(rows)

topic_results_df = apply_renames(load_jsonl(TOPIC_CKPT), TOPIC_RENAME_MAP)
truth_results_df = apply_renames(load_jsonl(TRUTH_CKPT), TRUTH_RENAME_MAP)
abl_raw_df = load_jsonl(ABLATION_CKPT)
llm_cmp_topic_df = apply_renames(load_jsonl(LLM_CMP_TOPIC_CKPT), TOPIC_RENAME_MAP)
llm_cmp_truth_df = apply_renames(load_jsonl(LLM_CMP_TRUTH_CKPT), TRUTH_RENAME_MAP)

DATA_PATH = Path("data")
with open(DATA_PATH / "topics.json", "r", encoding="utf-8") as f:
    topic_name_to_id = json.load(f)
topic_id_to_name = {int(v): k for k, v in topic_name_to_id.items()}

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "figure.dpi": 200,
})


# ════════════════════════════════════════════════════════════════════
# FIGURE 1: Pipeline Configurations (multi-panel)
# ════════════════════════════════════════════════════════════════════
def draw_pipeline_configs():
    fig, axes = plt.subplots(5, 1, figsize=(10, 14))
    fig.subplots_adjust(hspace=0.5)

    c_input = "#E8F4FD"
    c_embed = "#D5E8D4"
    c_db = "#F4ECF7"
    c_rerank = "#FDF2E9"
    c_llm = "#FDEDEC"
    c_output = "#E8F8F5"

    e_input = "#2C3E50"
    e_embed = "#27AE60"
    e_db = "#8E44AD"
    e_rerank = "#A0522D"
    e_llm = "#C0392B"
    e_output = "#1ABC9C"

    def draw_box(ax, x, y, text, fc, ec, fontsize=9, bold=True):
        weight = "bold" if bold else "normal"
        ax.text(x, y, text, ha="center", va="center", fontsize=fontsize, fontweight=weight,
                bbox=dict(boxstyle="round,pad=0.35", facecolor=fc, edgecolor=ec, linewidth=1.5))

    def draw_arrow(ax, x0, y0, x1, y1, color="#2C3E50", style="-"):
        ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                    arrowprops=dict(arrowstyle="->,head_width=0.15,head_length=0.1",
                                    color=color, lw=1.5, linestyle=style))

    # ── Panel A: Topic - Embedding Only ──
    ax = axes[0]
    ax.set_xlim(0, 10); ax.set_ylim(0, 2); ax.axis("off")
    ax.set_title("(a) Topic Classification: Embedding Only", fontsize=11, fontweight="bold", loc="left")
    draw_box(ax, 1.0, 1.0, "Medical\nStatement", c_input, e_input)
    draw_box(ax, 3.0, 1.0, "ModernBERT\nEmbedding", c_embed, e_embed)
    draw_box(ax, 5.2, 1.0, "ChromaDB\nRetrieve Top-k", c_db, e_db)
    draw_box(ax, 7.8, 1.0, "Topic from\nChunk Metadata\n(top-1 or voting)", c_output, e_output)
    draw_arrow(ax, 1.7, 1.0, 2.2, 1.0)
    draw_arrow(ax, 3.75, 1.0, 4.3, 1.0)
    draw_arrow(ax, 6.15, 1.0, 6.8, 1.0)

    # ── Panel B: Topic - Reranked ──
    ax = axes[1]
    ax.set_xlim(0, 10); ax.set_ylim(0, 2); ax.axis("off")
    ax.set_title("(b) Topic Classification: Reranked", fontsize=11, fontweight="bold", loc="left")
    draw_box(ax, 0.8, 1.0, "Medical\nStatement", c_input, e_input)
    draw_box(ax, 2.6, 1.0, "ModernBERT\nEmbedding", c_embed, e_embed)
    draw_box(ax, 4.6, 1.0, "ChromaDB\nRetrieve Top-k", c_db, e_db)
    draw_box(ax, 6.5, 1.0, "MiniLM\nReranker", c_rerank, e_rerank)
    draw_box(ax, 8.7, 1.0, "Topic from\nScore Aggregation\n(top-1 or sum)", c_output, e_output)
    draw_arrow(ax, 1.45, 1.0, 1.85, 1.0)
    draw_arrow(ax, 3.3, 1.0, 3.85, 1.0)
    draw_arrow(ax, 5.25, 1.0, 5.7, 1.0)
    draw_arrow(ax, 7.3, 1.0, 7.7, 1.0)

    # ── Panel C: Topic - LLM-based ──
    ax = axes[2]
    ax.set_xlim(0, 10); ax.set_ylim(0, 2); ax.axis("off")
    ax.set_title("(c) Topic Classification: LLM Source Selection", fontsize=11, fontweight="bold", loc="left")
    draw_box(ax, 0.65, 1.0, "Medical\nStatement", c_input, e_input)
    draw_box(ax, 2.3, 1.0, "ModernBERT\nEmbedding", c_embed, e_embed)
    draw_box(ax, 4.0, 1.0, "ChromaDB\nRetrieve Top-k", c_db, e_db)
    draw_box(ax, 5.7, 1.0, "MiniLM\nReranker", c_rerank, e_rerank)
    draw_box(ax, 7.4, 1.0, "Qwen3 LLM\n(pick source)", c_llm, e_llm)
    draw_box(ax, 9.2, 1.0, "Topic from\nChosen Chunk", c_output, e_output)
    draw_arrow(ax, 1.3, 1.0, 1.6, 1.0)
    draw_arrow(ax, 3.0, 1.0, 3.3, 1.0)
    draw_arrow(ax, 4.65, 1.0, 4.95, 1.0)
    draw_arrow(ax, 6.4, 1.0, 6.6, 1.0)
    draw_arrow(ax, 8.2, 1.0, 8.45, 1.0)

    # ── Panel D: Truth - Embedding + LLM ──
    ax = axes[3]
    ax.set_xlim(0, 10); ax.set_ylim(0, 2); ax.axis("off")
    ax.set_title("(d) Truth Prediction: Embedding + LLM (no reranking)", fontsize=11, fontweight="bold", loc="left")
    draw_box(ax, 1.0, 1.0, "Medical\nStatement", c_input, e_input)
    draw_box(ax, 3.0, 1.0, "ModernBERT\nEmbedding", c_embed, e_embed)
    draw_box(ax, 5.2, 1.0, "ChromaDB\nRetrieve Top-k", c_db, e_db)
    draw_box(ax, 7.5, 1.0, "Qwen3 LLM\n(fact-check)", c_llm, e_llm)
    draw_box(ax, 9.3, 1.0, "True /\nFalse", c_output, e_output)
    draw_arrow(ax, 1.7, 1.0, 2.2, 1.0)
    draw_arrow(ax, 3.75, 1.0, 4.3, 1.0)
    draw_arrow(ax, 6.1, 1.0, 6.6, 1.0)
    draw_arrow(ax, 8.35, 1.0, 8.7, 1.0)

    # ── Panel E: Truth - Reranked + LLM ──
    ax = axes[4]
    ax.set_xlim(0, 10); ax.set_ylim(0, 2); ax.axis("off")
    ax.set_title("(e) Truth Prediction: Reranked + LLM", fontsize=11, fontweight="bold", loc="left")
    draw_box(ax, 0.65, 1.0, "Medical\nStatement", c_input, e_input)
    draw_box(ax, 2.3, 1.0, "ModernBERT\nEmbedding", c_embed, e_embed)
    draw_box(ax, 4.0, 1.0, "ChromaDB\nRetrieve Top-k", c_db, e_db)
    draw_box(ax, 5.7, 1.0, "MiniLM\nReranker", c_rerank, e_rerank)
    draw_box(ax, 7.5, 1.0, "Qwen3 LLM\n(fact-check)", c_llm, e_llm)
    draw_box(ax, 9.3, 1.0, "True /\nFalse", c_output, e_output)
    draw_arrow(ax, 1.3, 1.0, 1.6, 1.0)
    draw_arrow(ax, 3.0, 1.0, 3.3, 1.0)
    draw_arrow(ax, 4.65, 1.0, 4.95, 1.0)
    draw_arrow(ax, 6.45, 1.0, 6.6, 1.0)
    draw_arrow(ax, 8.35, 1.0, 8.7, 1.0)

    plt.tight_layout()
    fig.savefig(FIG_DIR / "pipeline_configs.png", dpi=200, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: {FIG_DIR / 'pipeline_configs.png'}")

draw_pipeline_configs()


# ════════════════════════════════════════════════════════════════════
# FIGURE 2: Topic Accuracy and Latency
# ════════════════════════════════════════════════════════════════════
def draw_topic_bars():
    topic_metric_rows = []
    for strat, grp in topic_results_df.groupby("strategy"):
        acc = grp["is_correct"].mean() * 100
        topic_metric_rows.append({
            "strategy": strat,
            "topic_acc_%": round(acc, 2),
            "avg_latency_ms": round(grp["latency_ms"].mean(), 1),
        })
    df = pd.DataFrame(topic_metric_rows).sort_values("topic_acc_%", ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    colors = []
    for s in df["strategy"]:
        if "llm" in s:
            colors.append("#C0392B")
        elif "rerank" in s:
            colors.append("#E67E22")
        else:
            colors.append("#2E86C1")

    axes[0].barh(df["strategy"], df["topic_acc_%"], color=colors, edgecolor="white", linewidth=0.5)
    axes[0].set_xlabel("Topic Accuracy (%)")
    axes[0].set_title("Topic Accuracy by Strategy")
    axes[0].set_xlim(0, 100)
    axes[0].invert_yaxis()
    for i, v in enumerate(df["topic_acc_%"]):
        axes[0].text(v + 0.8, i, f"{v:.1f}%", va="center", fontsize=8)

    axes[1].barh(df["strategy"], df["avg_latency_ms"], color=colors, edgecolor="white", linewidth=0.5)
    axes[1].set_xlabel("Avg Latency (ms)")
    axes[1].set_title("Latency by Strategy")
    axes[1].set_xscale("log")
    axes[1].invert_yaxis()

    legend_elements = [
        mpatches.Patch(facecolor="#2E86C1", label="Embedding only"),
        mpatches.Patch(facecolor="#E67E22", label="Reranked"),
        mpatches.Patch(facecolor="#C0392B", label="LLM-based"),
    ]
    axes[0].legend(handles=legend_elements, loc="lower right", fontsize=8)

    plt.tight_layout()
    fig.savefig(FIG_DIR / "topic_accuracy_latency.png", dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {FIG_DIR / 'topic_accuracy_latency.png'}")

draw_topic_bars()


# ════════════════════════════════════════════════════════════════════
# FIGURE 3: Truth Prediction Bars
# ════════════════════════════════════════════════════════════════════
def draw_truth_bars():
    truth_metric_rows = []
    for strat, grp in truth_results_df.groupby("strategy"):
        y_true = grp["gt_is_true"].values
        y_pred = grp["pred_is_true"].values
        truth_metric_rows.append({
            "strategy": strat,
            "truth_acc_%": round((y_true == y_pred).mean() * 100, 2),
            "f1": round(f1_score(y_true, y_pred, zero_division=0), 4),
            "avg_latency_ms": round(grp["latency_ms"].mean(), 1),
        })
    df = pd.DataFrame(truth_metric_rows).sort_values("f1", ascending=False)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    colors = ["#E67E22" if "rerank" in s else "#2E86C1" for s in df["strategy"]]

    for ax, col, title, xlim in zip(
        axes,
        ["truth_acc_%", "f1", "avg_latency_ms"],
        ["Truth Accuracy (%)", "F1-Score", "Avg Latency (ms)"],
        [(70, 90), (0.7, 0.9), None],
    ):
        bars = ax.barh(df["strategy"], df[col], color=colors, edgecolor="white", linewidth=0.5)
        ax.set_xlabel(title)
        ax.set_title(title)
        if xlim:
            ax.set_xlim(*xlim)
        ax.invert_yaxis()

    legend_elements = [
        mpatches.Patch(facecolor="#2E86C1", label="Embedding + LLM"),
        mpatches.Patch(facecolor="#E67E22", label="Reranked + LLM"),
    ]
    axes[0].legend(handles=legend_elements, loc="lower right", fontsize=8)

    plt.tight_layout()
    fig.savefig(FIG_DIR / "truth_accuracy_bars.png", dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {FIG_DIR / 'truth_accuracy_bars.png'}")

draw_truth_bars()


# ════════════════════════════════════════════════════════════════════
# FIGURE 4: Pareto Plots
# ════════════════════════════════════════════════════════════════════
def draw_pareto():
    topic_rows = []
    for strat, grp in topic_results_df.groupby("strategy"):
        topic_rows.append({
            "strategy": strat,
            "topic_acc_%": round(grp["is_correct"].mean() * 100, 2),
            "avg_latency_ms": round(grp["latency_ms"].mean(), 1),
        })
    topic_df = pd.DataFrame(topic_rows)

    truth_rows = []
    for strat, grp in truth_results_df.groupby("strategy"):
        y_true = grp["gt_is_true"].values
        y_pred = grp["pred_is_true"].values
        truth_rows.append({
            "strategy": strat,
            "f1": round(f1_score(y_true, y_pred, zero_division=0), 4),
            "avg_latency_ms": round(grp["latency_ms"].mean(), 1),
        })
    truth_df = pd.DataFrame(truth_rows)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    ax = axes[0]
    for _, row in topic_df.iterrows():
        s = row["strategy"]
        if "llm" in s:
            color, marker = "#C0392B", "D"
        elif "rerank" in s:
            color, marker = "#E67E22", "s"
        else:
            color, marker = "#2E86C1", "o"
        ax.scatter(row["avg_latency_ms"], row["topic_acc_%"],
                   c=color, marker=marker, s=120, zorder=5, edgecolors="black", linewidths=0.5)
        ax.annotate(s, (row["avg_latency_ms"], row["topic_acc_%"]),
                    textcoords="offset points", xytext=(6, 6), fontsize=7)
    ax.set_xscale("log")
    ax.set_xlabel("Avg Latency (ms, log scale)")
    ax.set_ylabel("Topic Accuracy (%)")
    ax.set_title("Topic: Accuracy vs Latency")
    ax.set_ylim(70, 92)
    ax.grid(True, alpha=0.3)
    legend_elements = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor="#2E86C1", markersize=10, label="Embedding only"),
        Line2D([0], [0], marker="s", color="w", markerfacecolor="#E67E22", markersize=10, label="Reranked"),
        Line2D([0], [0], marker="D", color="w", markerfacecolor="#C0392B", markersize=10, label="LLM-based"),
    ]
    ax.legend(handles=legend_elements, loc="lower right", fontsize=8)

    ax = axes[1]
    for _, row in truth_df.iterrows():
        s = row["strategy"]
        color, marker = ("#E67E22", "s") if "rerank" in s else ("#2E86C1", "o")
        ax.scatter(row["avg_latency_ms"], row["f1"],
                   c=color, marker=marker, s=120, zorder=5, edgecolors="black", linewidths=0.5)
        ax.annotate(s, (row["avg_latency_ms"], row["f1"]),
                    textcoords="offset points", xytext=(6, 6), fontsize=7)
    ax.set_xlabel("Avg Latency (ms)")
    ax.set_ylabel("F1-Score")
    ax.set_title("Truth: F1 vs Latency")
    ax.set_ylim(0.7, 0.9)
    ax.grid(True, alpha=0.3)
    legend_elements2 = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor="#2E86C1", markersize=10, label="Embedding + LLM"),
        Line2D([0], [0], marker="s", color="w", markerfacecolor="#E67E22", markersize=10, label="Reranked + LLM"),
    ]
    ax.legend(handles=legend_elements2, loc="lower right", fontsize=8)

    plt.suptitle("Pareto Analysis: Accuracy vs Latency Trade-off", fontsize=13, y=1.02)
    plt.tight_layout()
    fig.savefig(FIG_DIR / "pareto.png", dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {FIG_DIR / 'pareto.png'}")

draw_pareto()


# ════════════════════════════════════════════════════════════════════
# FIGURE 5: Ablation Bar Chart
# ════════════════════════════════════════════════════════════════════
def draw_ablation():
    abl_df = (
        abl_raw_df
        .groupby(["chunk_size", "chunk_overlap", "num_chunks"])
        .agg(
            topic_acc_pct=("is_correct", lambda x: round(x.mean() * 100, 2)),
            avg_latency_ms=("latency_ms", lambda x: round(x.mean(), 1)),
            n=("is_correct", "count"),
        )
        .reset_index()
        .rename(columns={"topic_acc_pct": "topic_acc_%"})
    )

    fig, ax = plt.subplots(figsize=(10, 4))
    labels = [f"{int(r.chunk_size)}/{int(r.chunk_overlap)}" for r in abl_df.itertuples()]
    bars = ax.bar(labels, abl_df["topic_acc_%"], color="#2E86C1", edgecolor="white", linewidth=0.5)
    best_idx = abl_df["topic_acc_%"].idxmax()
    bars[best_idx].set_color("#C0392B")
    ax.set_title("Topic Accuracy vs Chunk Size/Overlap")
    ax.set_xlabel("chunk_size / chunk_overlap")
    ax.set_ylabel("Topic Accuracy (%)")
    ax.set_ylim(0, 100)
    for i, v in enumerate(abl_df["topic_acc_%"]):
        ax.text(i, v + 1.5, f"{v:.1f}%", ha="center", fontsize=9)
    plt.tight_layout()
    fig.savefig(FIG_DIR / "ablation.png", dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {FIG_DIR / 'ablation.png'}")

draw_ablation()


# ════════════════════════════════════════════════════════════════════
# FIGURE 6: LLM Comparison (Qwen3 vs Qwen3.5)
# ════════════════════════════════════════════════════════════════════
def draw_llm_comparison():
    baseline_truth = truth_results_df.copy()
    baseline_truth["model"] = "Qwen3-4B"
    comp_truth = llm_cmp_truth_df.copy()
    comp_truth["model"] = "Qwen3.5-4B"
    combined = pd.concat([baseline_truth, comp_truth], ignore_index=True)

    rows = []
    for (model, strat), grp in combined.groupby(["model", "strategy"]):
        y_true = grp["gt_is_true"].values
        y_pred = grp["pred_is_true"].values
        rows.append({
            "model": model,
            "strategy": strat,
            "truth_acc_%": round((y_true == y_pred).mean() * 100, 2),
            "f1": round(f1_score(y_true, y_pred, zero_division=0), 4),
            "recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
        })
    df = pd.DataFrame(rows)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    pivot_f1 = df.pivot(index="strategy", columns="model", values="f1")
    pivot_f1 = pivot_f1[["Qwen3-4B", "Qwen3.5-4B"]]
    pivot_f1.plot(kind="barh", ax=axes[0], color=["#2E86C1", "#C0392B"], edgecolor="white", linewidth=0.5)
    axes[0].set_xlabel("F1-Score")
    axes[0].set_title("Truth F1 by Strategy and Model")
    axes[0].set_xlim(0, 1.05)
    axes[0].legend(title="Model", fontsize=8)

    pivot_recall = df.pivot(index="strategy", columns="model", values="recall")
    pivot_recall = pivot_recall[["Qwen3-4B", "Qwen3.5-4B"]]
    pivot_recall.plot(kind="barh", ax=axes[1], color=["#2E86C1", "#C0392B"], edgecolor="white", linewidth=0.5)
    axes[1].set_xlabel("Recall")
    axes[1].set_title("Truth Recall by Strategy and Model")
    axes[1].set_xlim(0, 1.05)
    axes[1].legend(title="Model", fontsize=8)

    plt.tight_layout()
    fig.savefig(FIG_DIR / "llm_comparison.png", dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {FIG_DIR / 'llm_comparison.png'}")

draw_llm_comparison()


# ════════════════════════════════════════════════════════════════════
# FIGURE 7: Top Misclassified Topics
# ════════════════════════════════════════════════════════════════════
def draw_misclassified():
    best_strat = "topic_rerank_k25_top1"
    best_grp = topic_results_df[topic_results_df["strategy"] == best_strat].copy()
    errors = best_grp[~best_grp["is_correct"]].copy()
    errors["gt_topic_name"] = errors["gt_topic_id"].map(topic_id_to_name)

    topic_totals = best_grp.groupby("gt_topic_id").size().reset_index(name="total")
    topic_error_counts = (
        errors.groupby(["gt_topic_id", "gt_topic_name"])
        .size().reset_index(name="num_errors")
        .sort_values("num_errors", ascending=False)
    )
    topic_error_counts = topic_error_counts.merge(topic_totals, on="gt_topic_id")
    top = topic_error_counts.head(10)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(top["gt_topic_name"], top["num_errors"], color="#E67E22", edgecolor="white", linewidth=0.5)
    ax.set_xlabel("Number of Misclassifications")
    ax.set_title(f"Top 10 Most Misclassified Topics ({best_strat})")
    ax.invert_yaxis()
    for i, row in enumerate(top.itertuples()):
        ax.text(row.num_errors + 0.05, i, f"{row.num_errors}/{row.total}", va="center", fontsize=9)
    plt.tight_layout()
    fig.savefig(FIG_DIR / "misclassified_topics.png", dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {FIG_DIR / 'misclassified_topics.png'}")

draw_misclassified()


print("\n" + "=" * 60)
print("ALL FIGURES SAVED TO:", FIG_DIR)
print("=" * 60)
for f in sorted(FIG_DIR.glob("*.png")):
    print(f"  {f.name}")

## 13. State-of-the-Art LLM Comparison: Claude API

The previous sections benchmarked local 4B-parameter models (Qwen3 and Qwen3.5). Here we test whether a state-of-the-art commercial LLM can extract better answers from the same retrieved evidence.

We keep the retrieval pipeline identical and only swap the final reasoning step to Claude via the Anthropic API. This isolates the effect of LLM capability on truth prediction.

Two configurations are tested:
- ``truth_embed_k5``: Feed the best 5 passages to Claude -- directly comparable to the same strategy with Qwen.
- ``truth_embed_k10``: Feed the best 10 passages to Claude -- testing whether a stronger LLM can leverage more context.

In [ ]:
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import anthropic

load_dotenv()

CLAUDE_MODEL = "claude-sonnet-4-6"

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
if not ANTHROPIC_API_KEY:
    logger.warning("ANTHROPIC_API_KEY not set.")

CLAUDE_MAX_TOKENS = 100

# Avoid rate limits:
CLAUDE_RATE_LIMIT_DELAY = 0.5 

CLAUDE_TRUTH_CKPT = RESULTS_DIR / "claude_truth_results.jsonl"

logger.info(f"Claude model: {CLAUDE_MODEL})\n")
logger.info(f"API key set: {'Yes' if ANTHROPIC_API_KEY else 'NO'}\n")

In [ ]:
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)


class FactCheckResult(BaseModel):
    statement_is_true: int = Field(description="1 if the statement is true, 0 if false")
    source_chunk_id: int = Field(description="Index of the chunk containing the most relevant evidence")


def claude_classify(query, docs, model=None, max_tokens=None):
    """
    Send a fact-checking request to Claude with structured output.

    Input:  query (str), docs (list[str])
    Output: {"is_true": 0|1, "source_chunk_id": int, "raw": str}
    """
    model = model or CLAUDE_MODEL
    max_tokens = max_tokens or CLAUDE_MAX_TOKENS

    if not docs:
        return {"is_true": 0, "source_chunk_id": -1, "raw": ""}

    # Build prompt
    context_parts = []
    for i, chunk_text in enumerate(docs):
        context_parts.append(f"[CHUNK {i} below]\n{chunk_text}\n[CHUNK {i} above]")
    context = "\n\n---\n\n".join(context_parts)

    system_prompt = build_system_prompt(docs)
    user_prompt = f"""<context_passages>
{context}
</context_passages>

<medical_statement>
{query}
</medical_statement>

Evaluate if the statement is true or false based on the context and identify the single most relevant source chunk ID."""

    try:
        response = anthropic_client.messages.parse(
            model=model,
            max_tokens=max_tokens,
            system=system_prompt,
            messages=[
                {"role": "user", "content": user_prompt},
            ],
            output_format=FactCheckResult,
        )
    except anthropic.RateLimitError as e:
        logger.warning(f"Claude rate limited: {e}\n")
        return {"is_true": 0, "source_chunk_id": -1, "raw": "rate_limited"}
    except anthropic.APIError as e:
        logger.warning(f"Claude API error: {e}\n")
        return {"is_true": 0, "source_chunk_id": -1, "raw": str(e)}
    except Exception as e:
        logger.warning(f"Claude unexpected error: {e}\n")
        return {"is_true": 0, "source_chunk_id": -1, "raw": str(e)}

    parsed = response.parsed_output
    raw_text = response.content[0].text if response.content else ""

    if parsed is None:
        logger.warning(f"Claude returned no parsed output. Raw: {raw_text[:240]}\n")
        return {"is_true": 0, "source_chunk_id": -1, "raw": raw_text}

    return {
        "is_true": int(parsed.statement_is_true),
        "source_chunk_id": int(parsed.source_chunk_id),
        "raw": raw_text,
    }

In [ ]:
def _retrieve_embed_claude(query, k=5):
    """Retrieve by embedding only (no reranking), then call Claude."""
    docs, metas, _ = _retrieve(query, k=k)
    if not docs:
        return {"is_true": 0, "source_chunk_id": -1, "raw": ""}, []

    try:
        pred = claude_classify(query=query, docs=docs)
    except Exception as e:
        logger.warning(f"Claude inference failed: {e}\n")
        pred = {"is_true": 0, "source_chunk_id": -1, "raw": ""}

    return pred, metas


def truth_claude_embed_top5(query):
    pred, _ = _retrieve_embed_claude(query, k=5)
    return {"is_true": pred.get("is_true", 0), "raw": pred.get("raw", "")}


def truth_claude_embed_top10(query):
    pred, _ = _retrieve_embed_claude(query, k=10)
    return {"is_true": pred.get("is_true", 0), "raw": pred.get("raw", "")}


CLAUDE_TRUTH_STRATEGIES = [
    ("claude_embed_top5",  truth_claude_embed_top5),
    ("claude_embed_top10", truth_claude_embed_top10),
]

logger.info(f"Defined {len(CLAUDE_TRUTH_STRATEGIES)} Claude truth strategies.\n")

In [ ]:
claude_ckpt = Checkpoint(
    CLAUDE_TRUTH_CKPT,
    key_fields=["model", "statement_id", "strategy"],
)
logger.info(
    f"Claude truth benchmark: {len(CLAUDE_TRUTH_STRATEGIES)} strategies x "
    f"{len(gt_df)} statements. Already done: {claude_ckpt.count()} rows.\n"
)

for _, row in tqdm(gt_df.iterrows(), total=len(gt_df), desc=f"Claude ({CLAUDE_MODEL})"):
    query    = str(row["query"])
    gt_truth = int(row["is_true"])
    gt_topic = int(row["topic_id"])
    sid      = str(row["statement_id"])

    for strat_name, strat_fn in CLAUDE_TRUTH_STRATEGIES:
        if claude_ckpt.done(
            model=CLAUDE_MODEL, statement_id=sid, strategy=strat_name
        ):
            continue

        t0 = time.perf_counter()
        try:
            pred = strat_fn(query)
        except Exception as e:
            logger.warning(f"[{strat_name}] {sid} failed: {e}\n")
            pred = {"is_true": 0, "raw": ""}
        latency = (time.perf_counter() - t0) * 1000.0

        pred_truth = int(pred.get("is_true", 0))
        claude_ckpt.write({
            "model":         CLAUDE_MODEL,
            "statement_id":  sid,
            "strategy":      strat_name,
            "gt_topic_id":   gt_topic,
            "gt_is_true":    gt_truth,
            "pred_is_true":  pred_truth,
            "is_correct":    pred_truth == gt_truth,
            "latency_ms":    round(latency, 3),
            "raw_pred":      str(pred.get("raw", "")),
        })

        # Rate limit
        time.sleep(CLAUDE_RATE_LIMIT_DELAY)

claude_results_df = apply_renames(claude_ckpt.to_dataframe(), TRUTH_RENAME_MAP)
logger.info(f"Claude benchmark complete: {len(claude_results_df)} rows total.\n")
display(claude_results_df.head(10))

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from matplotlib.patches import Patch

# Load all truth results: Qwen3 baseline + Qwen3.5 + Claude

# Qwen3 baseline
baseline_truth_df = apply_renames(truth_ckpt.to_dataframe(), TRUTH_RENAME_MAP).copy()
baseline_truth_df["model"] = LLM_MODEL_ID.split("/")[-1]

# Qwen3.5 comparison
if LLM_CMP_TRUTH_CKPT.exists():
    qwen35_truth_df = apply_renames(
        Checkpoint(LLM_CMP_TRUTH_CKPT, key_fields=["model", "statement_id", "strategy"]).to_dataframe(),
        TRUTH_RENAME_MAP,
    )
else:
    qwen35_truth_df = pd.DataFrame()

# Claude
claude_truth_df = claude_results_df.copy()

# Compute metrics for each (model, strategy) pair
all_truth = pd.concat(
    [baseline_truth_df, qwen35_truth_df, claude_truth_df],
    ignore_index=True,
)

metric_rows = []
for (model, strat), grp in all_truth.groupby(["model", "strategy"]):
    y_true = grp["gt_is_true"].values
    y_pred = grp["pred_is_true"].values
    try:
        auroc = roc_auc_score(y_true, y_pred)
    except Exception:
        auroc = float("nan")
    metric_rows.append({
        "model":          model,
        "strategy":       strat,
        "n":              len(grp),
        "truth_acc_%":    round((y_true == y_pred).mean() * 100, 2),
        "precision":      round(precision_score(y_true, y_pred, zero_division=0), 4),
        "recall":         round(recall_score(y_true, y_pred, zero_division=0), 4),
        "f1":             round(f1_score(y_true, y_pred, zero_division=0), 4),
        "auroc":          round(auroc, 4),
        "avg_latency_ms": round(grp["latency_ms"].mean(), 1),
    })

all_metrics_df = pd.DataFrame(metric_rows).sort_values(
    ["strategy", "f1"], ascending=[True, False]
)

print("=" * 70)
print("  FULL MODEL COMPARISON -- Truth Prediction (all models & strategies)")
print("=" * 70)
display(all_metrics_df)

# Side-by-side: embed_k5 across all models
top5_comparison = all_metrics_df[
    all_metrics_df["strategy"] == "truth_embed_k5"
].copy()

if not top5_comparison.empty:
    print("\n" + "=" * 70)
    print("  HEAD-TO-HEAD -- truth_embed_k5 (same retrieval, different LLM)")
    print("=" * 70)
    display(
        top5_comparison[["model", "strategy", "truth_acc_%", "f1", "precision", "recall", "auroc"]]
        .sort_values("f1", ascending=False)
    )

# Claude-only: k5 vs k10
claude_only = all_metrics_df[all_metrics_df["model"].str.contains("claude", case=False)].copy()

if not claude_only.empty:
    print("\n" + "=" * 70)
    print("  CLAUDE CONTEXT DEPTH -- Top-5 vs Top-10 Passages")
    print("=" * 70)
    display(
        claude_only[["model", "strategy", "truth_acc_%", "f1", "precision", "recall"]]
        .sort_values("f1", ascending=False)
    )

In [ ]:
from matplotlib.lines import Line2D

# Assign colors per model
model_colors = {}
palette = ["#2E86C1", "#E67E22", "#C0392B", "#27AE60", "#8E44AD"]
for i, model in enumerate(sorted(all_metrics_df["model"].unique())):
    model_colors[model] = palette[i % len(palette)]

# ── Grouped bar chart: bars grouped by strategy, colored by model ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Use all strategies that any model has results for
plot_df = all_metrics_df.copy()

if not plot_df.empty:
    pivot_f1 = plot_df.pivot(index="strategy", columns="model", values="f1")
    colors_ordered = [model_colors.get(m, "grey") for m in pivot_f1.columns]
    pivot_f1.plot(
        kind="barh", ax=axes[0], color=colors_ordered,
        edgecolor="white", linewidth=0.5,
    )
    axes[0].set_xlabel("F1-Score")
    axes[0].set_title("Truth F1 by Strategy & Model")
    axes[0].set_xlim(0, 1.05)
    axes[0].legend(title="Model", fontsize=8, loc="lower right")

    pivot_acc = plot_df.pivot(index="strategy", columns="model", values="truth_acc_%")
    pivot_acc.plot(
        kind="barh", ax=axes[1], color=colors_ordered,
        edgecolor="white", linewidth=0.5,
    )
    axes[1].set_xlabel("Truth Accuracy (%)")
    axes[1].set_title("Truth Accuracy by Strategy & Model")
    axes[1].set_xlim(0, 105)
    axes[1].legend(title="Model", fontsize=8, loc="lower right")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "claude_comparison_bars.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {RESULTS_DIR / 'claude_comparison_bars.png'}")

# Overall model summary
overall_rows = []
for model, grp in all_truth.groupby("model"):
    y_true = grp["gt_is_true"].values
    y_pred = grp["pred_is_true"].values
    overall_rows.append({
        "model": model,
        "strategies": grp["strategy"].nunique(),
        "mean_truth_acc_%": round((y_true == y_pred).mean() * 100, 2),
        "mean_f1": round(f1_score(y_true, y_pred, zero_division=0), 4),
        "mean_precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "mean_recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
        "avg_latency_ms": round(grp["latency_ms"].mean(), 1),
    })

overall_df = pd.DataFrame(overall_rows).sort_values("mean_f1", ascending=False)
print("\n" + "=" * 70)
print("  OVERALL MODEL SUMMARY (across all strategies per model)")
print("=" * 70)
display(overall_df)

# ── Precision vs Recall scatter (zoomed to cluster) ──
fig2, ax2 = plt.subplots(figsize=(8, 6))
for _, row in all_metrics_df.iterrows():
    model = row["model"]
    color = model_colors.get(model, "grey")
    marker = "D" if "claude" in model.lower() else ("s" if "3.5" in model else "o")
    ax2.scatter(
        row["recall"], row["precision"],
        c=color, marker=marker, s=120, zorder=5,
        edgecolors="black", linewidths=0.5,
    )
    label = f"{row['strategy']}\n({model})"
    ax2.annotate(
        label,
        (row["recall"], row["precision"]),
        textcoords="offset points", xytext=(6, 6), fontsize=7,
    )

ax2.set_xlabel("Recall")
ax2.set_ylabel("Precision")
ax2.set_title("Precision vs Recall -- All Models & Strategies")
# Zoom into the cluster for readability
ax2.set_xlim(0.5, 1.0)
ax2.set_ylim(0.88, 1.02)
ax2.grid(True, alpha=0.2)

legend_handles = [
    Line2D([0], [0], marker="o", color="w",
           markerfacecolor=model_colors.get(m, "grey"), markersize=10, label=m)
    for m in sorted(all_metrics_df["model"].unique())
]
ax2.legend(handles=legend_handles, loc="lower left", fontsize=8, title="Model")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "claude_precision_recall.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {RESULTS_DIR / 'claude_precision_recall.png'}")

In [ ]:
# Save Claude results to Parquet
claude_parquet_path = RESULTS_DIR / "claude_truth_results.parquet"
claude_results_df.to_parquet(claude_parquet_path, index=False)
logger.info(f"Saved {len(claude_results_df)} rows -> {claude_parquet_path}\n")

# Also save the combined metrics table
all_metrics_path = RESULTS_DIR / "all_models_truth_metrics.parquet"
all_metrics_df.to_parquet(all_metrics_path, index=False)
logger.info(f"Saved combined metrics -> {all_metrics_path}\n")

logger.info("Section 13 complete.\n")